# Semantic Model Similarity - Results

Interactive review of schema, security, and combined similarity, plus directional schema coverage. This notebook reads saved outputs from **001_semantic_model_similarity** and shows security-definition differences separately from schema and report evidence.

Run **001 -> 002** against the same Lakehouse after upgrading or changing model security. Older or inconsistent results keep their schema scores but cannot supply a security-inclusive combined score. Matching security definitions do not establish matching role assignments or effective user access.

In [ ]:
def build_report_dependency_payload(models, reports, scans, load_status="available"):
    from collections import Counter
    from urllib.parse import quote

    result = {
        "status": "not_collected",
        "scannedAt": "",
        "scope": "",
        "workspaceCount": 0,
        "incompleteWorkspaces": 0,
        "reportCount": 0,
        "linkedReportCount": 0,
        "byModel": {model_id: [] for model_id in models},
        "otherReports": [],
        "scans": [],
    }
    if load_status != "available":
        result["status"] = load_status
        return result
    if not scans:
        return result
    scan_ids = {row.get("scan_id") for row in scans}
    scan_workspaces = {row.get("report_workspace_id") for row in scans}
    scopes = {row.get("scan_scope") for row in scans}
    if (len(scan_ids) != 1 or not next(iter(scan_ids)) or len(scopes) != 1
            or len(scan_workspaces) != len(scans)
            or any(row.get("scan_id") not in scan_ids for row in reports)):
        result["status"] = "inconsistent"
        return result
    unique_reports = {}
    for row in reports:
        key = (row.get("report_workspace_id"), row.get("report_id"))
        if not all(key) or key[0] not in scan_workspaces:
            result["status"] = "inconsistent"
            return result
        if key in unique_reports and unique_reports[key].get("model_id") != row.get("model_id"):
            result["status"] = "inconsistent"
            return result
        unique_reports[key] = row
    workspace_counts = Counter(key[0] for key in unique_reports)
    if any(row.get("report_count") != workspace_counts[row.get("report_workspace_id")] for row in scans):
        result["status"] = "inconsistent"
        return result
    result["incompleteWorkspaces"] = sum(row.get("scan_status") != "complete" for row in scans)
    result["status"] = "partial" if result["incompleteWorkspaces"] else "complete"
    result["scannedAt"] = str(scans[0].get("scanned_at") or "")
    result["scope"] = str(scans[0].get("scan_scope") or "Unknown scope")
    result["workspaceCount"] = sum(bool(row.get("report_workspace_id")) for row in scans)
    result["scans"] = [{
        "workspace": str(row.get("report_workspace_name") or "Workspace discovery"),
        "status": str(row.get("scan_status") or "unknown"),
        "reportCount": int(row.get("report_count") or 0),
        "errorType": str(row.get("error_type") or ""),
    } for row in scans]
    model_index = {str(model_id).casefold(): model_id for model_id in models}
    for row in unique_reports.values():
        model_id = model_index.get(str(row.get("model_id") or "").casefold())
        workspace_id = str(row["report_workspace_id"])
        report_id = str(row["report_id"])
        report = {
            "id": report_id,
            "name": str(row.get("report_name") or report_id),
            "workspaceId": workspace_id,
            "workspace": str(row.get("report_workspace_name") or workspace_id),
            "url": "https://app.powerbi.com/groups/" + quote(workspace_id, safe="") + "/reports/" + quote(report_id, safe=""),
            "modelId": model_id or "",
            "crossWorkspace": False,
        }
        if model_id is not None and row.get("report_type") == "PowerBIReport":
            report["crossWorkspace"] = workspace_id.casefold() != str(models[model_id].get("workspace_id") or "").casefold()
            result["byModel"][model_id].append(report)
            result["linkedReportCount"] += 1
        else:
            report["reason"] = {
                "missing_dataset_id": "Missing model binding",
                "unsupported_report_type": "Unsupported report type",
            }.get(row.get("binding_status"), "Model outside catalog")
            result["otherReports"].append(report)
    result["reportCount"] = len(unique_reports)
    for group in list(result["byModel"].values()) + [result["otherReports"]]:
        group.sort(key=lambda report: (report["name"].casefold(), report["workspace"].casefold(), report["id"]))
    return result


def build_security_results(model_rows, signature_rows, pair_rows, run_rows):
    import hashlib
    import json
    import math
    from collections import Counter, defaultdict

    def encoded(value):
        return json.dumps(value, sort_keys=True, separators=(",", ":"), ensure_ascii=True, allow_nan=False)

    def number(value):
        if isinstance(value, bool):
            return None
        try:
            numeric = float(value)
            return numeric if math.isfinite(numeric) and 0 <= numeric <= 1 else None
        except (TypeError, ValueError):
            return None

    result = {"models": {}, "pairs": [], "notice": "", "combinedWeights": None}
    meta = run_rows[0] if len(run_rows) == 1 else {}
    current_run = (meta.get("score_version") == 3 and isinstance(meta.get("analysis_run_id"), str)
                   and bool(meta["analysis_run_id"]) and meta.get("model_count") == len(signature_rows)
                   and meta.get("pair_count") == len(pair_rows))
    try:
        weights = json.loads(meta["combined_weights_json"])
        if set(weights) != {"schema", "security"} or any(number(value) is None for value in weights.values()):
            raise ValueError("Invalid combined weights")
        total = math.fsum(weights.values())
        if total <= 0:
            raise ValueError("Empty combined weights")
        weights = {key: value / total for key, value in weights.items()}
        result["combinedWeights"] = weights if current_run else None
    except (KeyError, TypeError, ValueError):
        current_run = False
        weights = {}
    saved_by_model, current_by_model = defaultdict(list), defaultdict(list)
    for row in signature_rows:
        saved_by_model[str(row.get("model_id"))].append(row)
    for row in model_rows:
        current_by_model[str(row.get("model_id"))].append(row)
    for model_id, identities in current_by_model.items():
        security = {"status": "not_collected", "fingerprint": None, "roles": [], "propagation": [], "roleCount": None}
        result["models"][model_id] = security
        saved = saved_by_model[model_id]
        if not saved:
            continue
        if not current_run or len(identities) != 1 or len(saved) != 1:
            security["status"] = "inconsistent"
            continue
        identity, snapshot = identities[0], saved[0]
        if (snapshot.get("score_version") != 3 or snapshot.get("security_schema_version") != 1
                or snapshot.get("analysis_run_id") != meta["analysis_run_id"]
                or not isinstance(identity.get("catalog_scan_id"), str) or not identity["catalog_scan_id"]
                or any(snapshot.get(key) != identity.get(key) for key in ("model_id", "workspace_id", "catalog_scan_id"))):
            security["status"] = "inconsistent"
            continue
        if snapshot.get("security_scan_status") != "complete":
            security["status"] = "incomplete"
            continue
        try:
            definition = json.loads(snapshot["security_definition_json"])
            roles, propagation = definition["roles"], definition["propagation"]
            if not isinstance(roles, list) or not isinstance(propagation, list) or snapshot.get("role_count") != len(roles):
                raise ValueError("Invalid security inventory")
            role_keys = []
            for role in roles:
                if not isinstance(role["name"], str) or not role["name"].strip():
                    raise ValueError("Missing role name")
                if role["model_permission"] not in ("None", "Read", "ReadRefresh", "Refresh", "Administrator"):
                    raise ValueError("Invalid model permission")
                bundle = {"model_permission": role["model_permission"]}
                for family, width in (("rls", 2), ("table_ols", 2), ("column_ols", 3)):
                    facts = role[family]
                    if not isinstance(facts, list) or any(not isinstance(fact, list) or len(fact) != width or not all(isinstance(value, str) for value in fact) for fact in facts):
                        raise ValueError("Invalid security facts")
                    bundle[family] = facts
                role_key = encoded(bundle)
                if hashlib.sha256(role_key.encode("utf-8")).hexdigest() != role["fingerprint"]:
                    raise ValueError("Role signature differs")
                role_keys.append(role_key)
            if any(not isinstance(fact, list) or len(fact) != 9 or type(fact[4]) is not bool
                   or not all(isinstance(value, str) for value in fact[:4] + fact[5:]) for fact in propagation):
                raise ValueError("Invalid propagation facts")
            canonical = encoded({"version": 1, "roles": sorted(role_keys), "propagation": sorted(propagation)})
            fingerprint = hashlib.sha256(canonical.encode("utf-8")).hexdigest()
            if fingerprint != snapshot["security_fingerprint"]:
                raise ValueError("Security signature differs")
            security.update(status="complete", fingerprint=fingerprint, roles=roles, propagation=propagation, roleCount=len(roles))
        except (KeyError, TypeError, ValueError):
            security["status"] = "inconsistent"
    pair_counts = Counter(tuple(sorted((str(row.get("model_id_a")), str(row.get("model_id_b"))))) for row in pair_rows)
    for row in pair_rows:
        new_format = row.get("score_version") == 3
        schema = number(row.get("schema_score") if new_format else row.get("composite_score"))
        fields = {"schema": schema, "security": None, "combined": None, "securityStatus": "unknown",
                  "scoreMode": "unavailable_security", "securityEvidence": {}, "scoreStatus": "legacy" if not new_format else "inconsistent"}
        result["pairs"].append(fields)
        if not new_format or not current_run or row.get("analysis_run_id") != meta["analysis_run_id"]:
            continue
        id_a, id_b = str(row["model_id_a"]), str(row["model_id_b"])
        if pair_counts[tuple(sorted((id_a, id_b)))] != 1:
            continue
        left, right = result["models"].get(id_a, {}), result["models"].get(id_b, {})
        if left.get("status") != "complete" or right.get("status") != "complete":
            fields["scoreStatus"] = "security_unavailable"
            continue
        if (row.get("security_fingerprint_a") != left["fingerprint"] or row.get("security_fingerprint_b") != right["fingerprint"]
                or row.get("catalog_scan_id_a") != current_by_model[id_a][0].get("catalog_scan_id")
                or row.get("catalog_scan_id_b") != current_by_model[id_b][0].get("catalog_scan_id")):
            continue
        expected_status = "not_applicable" if not left["roles"] and not right["roles"] else ("match" if left["fingerprint"] == right["fingerprint"] else "different")
        try:
            security_score, combined = number(row.get("security_score")), number(row.get("combined_score"))
            if schema is None or combined is None or row.get("security_comparison_status") != expected_status:
                raise ValueError("Missing or inconsistent scores")
            mode = "schema_only_no_security" if expected_status == "not_applicable" else "combined"
            if row.get("score_mode") != mode:
                raise ValueError("Inconsistent score mode")
            if expected_status == "not_applicable":
                if security_score is not None:
                    raise ValueError("Security must be not applicable")
                expected_combined = schema
            else:
                if security_score is None:
                    raise ValueError("Missing security score")
                expected_combined = math.fsum((weights["schema"] * schema, weights["security"] * security_score))
            if not math.isclose(combined, expected_combined, rel_tol=0, abs_tol=1e-12):
                raise ValueError("Combined score does not agree with components")
            evidence = json.loads(row["security_evidence_json"])
            if not isinstance(evidence, dict) or not isinstance(evidence["role_matches"], list):
                raise ValueError("Missing security evidence")
            matched_a, matched_b = set(), set()
            for match in evidence["role_matches"]:
                index_a, index_b = match["role_a"], match["role_b"]
                if (type(index_a) is not int or type(index_b) is not int or not 0 <= index_a < len(left["roles"])
                        or not 0 <= index_b < len(right["roles"]) or index_a in matched_a or index_b in matched_b
                        or number(match["score"]) is None):
                    raise ValueError("Invalid role alignment")
                matched_a.add(index_a)
                matched_b.add(index_b)
            if (set(evidence["unmatched_roles_a"]) != set(range(len(left["roles"]))) - matched_a
                    or set(evidence["unmatched_roles_b"]) != set(range(len(right["roles"]))) - matched_b):
                raise ValueError("Incomplete role alignment")
            fields.update(security=security_score, combined=combined, securityStatus=expected_status, scoreMode=mode, securityEvidence=evidence, scoreStatus="current")
        except (KeyError, TypeError, ValueError):
            continue
    if any(pair["scoreStatus"] in ("legacy", "inconsistent") for pair in result["pairs"]):
        result["notice"] = "Some saved scores lack aligned security evidence. Run 001 again; legacy composite values are shown only as schema scores."
    elif any(pair["securityStatus"] == "unknown" for pair in result["pairs"]):
        result["notice"] = "Security could not be fully assessed for some models. Their schema scores remain available; security and combined scores are unavailable."
    return result


def norm_m(value):
    # Power Query M normalized for comparison: comments removed, whitespace collapsed and case
    # folded outside string literals. Literals (server, database, path names) are kept verbatim.
    import re

    if value is None or (isinstance(value, float) and value != value):
        return ""
    pieces = []
    for match in re.finditer(r'"(?:[^"]|"")*"|/\*.*?\*/|//[^\n]*|[^"/]+|[/"]', str(value), flags=re.S):
        token = match.group(0)
        if token.startswith('"') and len(token) > 1:
            pieces.append(token)
        elif token.startswith("/*") or token.startswith("//"):
            pieces.append(" ")
        else:
            pieces.append(token.casefold())
    return re.sub(r"\s+", " ", "".join(pieces)).strip()


def render_results():
    """Load the latest results from the lakehouse and render the interactive app."""
    import hashlib
    import json
    import math
    import re
    from datetime import datetime, timezone

    import pandas as pd
    from pyspark.sql.utils import AnalysisException

    def load_delta(table_name):
        return spark.table(table_name).toPandas()

    models_df = load_delta("semantic_models")
    tables_df = load_delta("semantic_model_tables")
    columns_df = load_delta("semantic_model_columns")
    relationships_df = load_delta("semantic_model_relationships")
    measures_df = load_delta("semantic_model_measures")
    datasources_df = load_delta("semantic_model_datasources")
    pairs_df = load_delta("semantic_model_similarity_pairs")
    try:
        query_rows = load_delta("semantic_model_queries").to_dict("records")
    except AnalysisException:
        query_rows = []
    try:
        run_meta_df = load_delta("semantic_model_similarity_run")
    except AnalysisException:
        run_meta_df = pd.DataFrame()
    try:
        signature_rows = load_delta("semantic_model_signatures").to_dict("records")
    except AnalysisException:
        signature_rows = []

    report_rows, report_scans = [], []
    report_load_status = "available"
    try:
        report_rows = load_delta("semantic_model_report_dependencies").to_dict("records")
        report_scans = load_delta("semantic_model_report_scan").to_dict("records")
    except AnalysisException as error:
        error_class = getattr(error, "getErrorClass", lambda: "")()
        report_load_status = "not_collected" if error_class in ("TABLE_OR_VIEW_NOT_FOUND", "PATH_NOT_FOUND", "DELTA_PATH_DOES_NOT_EXIST") else "unavailable"

    if models_df.empty:
        raise ValueError(
            "No rows in semantic_models in the attached lakehouse. "
            "Run 001_semantic_model_similarity first."
        )

    def norm(value):
        if value is None or (isinstance(value, float) and pd.isna(value)):
            return ""
        return re.sub(r"\s+", " ", str(value)).strip().casefold()

    def norm_dax(value):
        if value is None or (isinstance(value, float) and pd.isna(value)):
            return ""
        text = re.sub(r"/\*.*?\*/", " ", str(value), flags=re.DOTALL)
        text = re.sub(r"//.*", " ", text)
        return re.sub(r"\s+", " ", text).strip().casefold()

    def disp(value):
        if value is None or (isinstance(value, float) and pd.isna(value)):
            return ""
        return str(value)

    def rounded(value):
        try:
            numeric = float(value)
            return round(numeric, 4) if math.isfinite(numeric) else None
        except (TypeError, ValueError):
            return None

    signatures = {}
    for _, row in models_df.iterrows():
        model_id = str(row["model_id"])
        signatures[model_id] = {
            "model_id": model_id,
            "workspace_id": disp(row.get("workspace_id")),
            "workspace_name": disp(row.get("workspace_name")),
            "model_name": disp(row.get("model_name")) or model_id,
        }

    report_payload = build_report_dependency_payload(signatures, report_rows, report_scans, report_load_status)
    pair_records = pairs_df.to_dict("records")
    security_context = build_security_results(models_df.to_dict("records"), signature_rows, pair_records, run_meta_df.to_dict("records"))

    if len(run_meta_df) == 1:
        meta = run_meta_df.iloc[0]
        duplicate_threshold = float(meta["duplicate_threshold"])
        similar_threshold = float(meta["similar_threshold"])
        containment_threshold = float(meta["containment_threshold"])
    else:
        duplicate_threshold, similar_threshold, containment_threshold = 0.95, 0.70, 0.95

    # Shared DAX pool: repeated expressions are serialized once.
    dax_pool = []
    dax_index = {}

    def dax_id(text):
        if not text:
            return -1
        idx = dax_index.get(text)
        if idx is None:
            idx = len(dax_pool)
            dax_pool.append(text)
            dax_index[text] = idx
        return idx

    # Shared Power Query pool, serialized once like DAX.
    m_pool = []
    m_index = {}

    def m_id(text):
        if not text:
            return -1
        idx = m_index.get(text)
        if idx is None:
            idx = len(m_pool)
            m_pool.append(text)
            m_index[text] = idx
        return idx

    def dax_hash(text):
        normalized = norm_dax(text)
        return hashlib.md5(normalized.encode("utf-8")).hexdigest()[:12] if normalized else ""

    # Inventories include every catalog model so direct Compare and the map cover the full comparable estate.
    inventories = {
        model_id: {
            "tables": {},
            "columns": {},
            "measures": {},
            "relationships": {},
            "datasources": {},
            "queries": {},
        }
        for model_id in signatures
    }

    for _, row in tables_df.iterrows():
        model_id = str(row["model_id"])
        if model_id in inventories:
            key = norm(row["table_name"])
            if key:
                inventories[model_id]["tables"].setdefault(key, disp(row["table_name"]))

    for _, row in columns_df.iterrows():
        model_id = str(row["model_id"])
        if model_id in inventories:
            table_key, column_key = norm(row["table_name"]), norm(row["column_name"])
            key = table_key + "." + column_key
            if column_key:
                inventories[model_id]["columns"].setdefault(
                    key,
                    {
                        "table": disp(row["table_name"]),
                        "tableKey": table_key,
                        "name": disp(row["column_name"]),
                    },
                )

    for _, row in measures_df.iterrows():
        model_id = str(row["model_id"])
        if model_id in inventories:
            key = norm(row["measure_name"])
            expression = disp(row.get("expression"))
            if key:
                inventories[model_id]["measures"].setdefault(
                    key,
                    {
                        "name": disp(row["measure_name"]),
                        "daxId": dax_id(expression),
                        "daxHash": dax_hash(expression),
                    },
                )

    for _, row in relationships_df.iterrows():
        model_id = str(row["model_id"])
        if model_id in inventories:
            key = (
                f"{norm(row['from_table'])}.{norm(row['from_column'])}"
                f"->{norm(row['to_table'])}.{norm(row['to_column'])}"
            )
            inventories[model_id]["relationships"].setdefault(
                key,
                {
                    "from": f"{disp(row['from_table'])}[{disp(row['from_column'])}]",
                    "to": f"{disp(row['to_table'])}[{disp(row['to_column'])}]",
                },
            )

    for _, row in datasources_df.iterrows():
        model_id = str(row["model_id"])
        if model_id in inventories:
            connection = (
                row.get("connection_string")
                or row.get("connection_details")
                or row.get("datasource_name")
            )
            key = norm(connection)
            if key:
                inventories[model_id]["datasources"].setdefault(key, disp(connection))

    m_partitions = {}
    for row in query_rows:
        if row.get("query_kind") == "partition" and norm_m(row.get("expression")):
            table_key = (str(row.get("model_id")), norm(row.get("table_name")))
            m_partitions[table_key] = m_partitions.get(table_key, 0) + 1
    for row in query_rows:
        model_id = str(row.get("model_id"))
        expression = disp(row.get("expression"))
        normalized = norm_m(expression)
        if model_id not in inventories or not normalized:
            continue
        if row.get("query_kind") == "shared_expression":
            key, name = "expression:" + norm(row.get("partition_name")), "Expression: " + disp(row.get("partition_name"))
        else:
            key, name = "table:" + norm(row.get("table_name")), disp(row.get("table_name"))
            # Tables with several M partitions compare each partition separately.
            if m_partitions.get((model_id, norm(row.get("table_name"))), 0) > 1:
                key, name = key + "/" + norm(row.get("partition_name")), name + " (" + disp(row.get("partition_name")) + ")"
        inventories[model_id]["queries"].setdefault(key, {
            "name": name,
            "mId": m_id(expression),
            "mHash": hashlib.md5(normalized.encode("utf-8")).hexdigest()[:12],
        })

    models_payload = {}
    for model_id, identity in signatures.items():
        inv = inventories[model_id]
        models_payload[model_id] = {
            "name": identity["model_name"],
            "workspace": identity["workspace_name"],
            "workspaceId": identity["workspace_id"],
            "tables": [{"key": k, "name": v} for k, v in inv["tables"].items()],
            "columns": [
                {
                    "key": k,
                    "table": v["table"],
                    "tableKey": v["tableKey"],
                    "name": v["name"],
                }
                for k, v in inv["columns"].items()
            ],
            "measures": [
                {
                    "key": k,
                    "name": v["name"],
                    "daxId": v["daxId"],
                    "daxHash": v["daxHash"],
                }
                for k, v in inv["measures"].items()
            ],
            "relationships": [
                {"key": k, "from": v["from"], "to": v["to"]}
                for k, v in inv["relationships"].items()
            ],
            "datasources": [{"key": k, "name": v} for k, v in inv["datasources"].items()],
            "queries": [{"key": k, **v} for k, v in inv["queries"].items()],
            "security": security_context["models"][model_id],
        }

    def shared_queries(value):
        try:
            count = float(value)
            return int(count) if math.isfinite(count) and count >= 0 else None
        except (TypeError, ValueError):
            return None

    def pair_object(row, security_fields):
        id_a, id_b = str(row["model_id_a"]), str(row["model_id_b"])
        sig_a, sig_b = signatures.get(id_a, {}), signatures.get(id_b, {})
        coverage_a_in_b = rounded(row.get("model_a_in_model_b"))
        coverage_b_in_a = rounded(row.get("model_b_in_model_a"))
        relationship = disp(row.get("containment_relationship"))

        contained_id = containing_id = ""
        contained_coverage = None
        if relationship == "model_a_contains_model_b":
            contained_id, containing_id, contained_coverage = id_b, id_a, coverage_b_in_a
        elif relationship == "model_b_contains_model_a":
            contained_id, containing_id, contained_coverage = id_a, id_b, coverage_a_in_b
        elif relationship == "equivalent":
            contained_coverage = max(coverage_a_in_b or 0, coverage_b_in_a or 0)
        elif (coverage_a_in_b or 0) >= (coverage_b_in_a or 0):
            contained_id, containing_id, contained_coverage = id_a, id_b, coverage_a_in_b
        else:
            contained_id, containing_id, contained_coverage = id_b, id_a, coverage_b_in_a

        return {
            "idA": id_a,
            "idB": id_b,
            "modelA": sig_a.get("model_name") or disp(row.get("model_a")) or id_a,
            "workspaceA": sig_a.get("workspace_name") or disp(row.get("workspace_a")),
            "modelB": sig_b.get("model_name") or disp(row.get("model_b")) or id_b,
            "workspaceB": sig_b.get("workspace_name") or disp(row.get("workspace_b")),
            "scored": True,
            **security_fields,
            "containment": rounded(row.get("containment_score")),
            "relationship": relationship,
            "containedId": contained_id,
            "containingId": containing_id,
            "containedCoverage": contained_coverage,
            "aInB": coverage_a_in_b,
            "bInA": coverage_b_in_a,
            "crossWorkspace": bool(row.get("cross_workspace", False)),
            "sameName": bool(row.get("same_model_name", False)),
            "jaccard": {
                "tables": rounded(row.get("jaccard_tables")),
                "columns": rounded(row.get("jaccard_columns")),
                "measures": rounded(row.get("jaccard_measure_names")),
                "relationships": rounded(row.get("jaccard_relationships")),
                "datasources": rounded(row.get("jaccard_datasources")),
            },
            "daxCosine": rounded(row.get("dax_embedding_cosine")),
            "powerQuery": rounded(row.get("power_query_similarity")),
            "powerQueryStatus": disp(row.get("power_query_status")) or "unknown",
            "sharedQueries": shared_queries(row.get("shared_query_count")),
        }

    all_pairs_payload = [pair_object(row, fields) for row, fields in zip(pair_records, security_context["pairs"])]
    model_list = sorted(
        [
            {
                "id": model_id,
                "name": model["name"],
                "workspace": model["workspace"],
                "workspaceId": model["workspaceId"],
            }
            for model_id, model in models_payload.items()
        ],
        key=lambda item: (item["name"].casefold(), item["workspace"].casefold()),
    )

    default_a = default_b = ""
    if all_pairs_payload:
        best_pair = max(all_pairs_payload, key=lambda pair: (pair["combined"] is not None, pair["combined"] if pair["combined"] is not None else (pair["schema"] or 0)))
        default_a, default_b = best_pair["idA"], best_pair["idB"]
    elif len(model_list) >= 2:
        default_a, default_b = model_list[0]["id"], model_list[1]["id"]

    app_data = {
        "generatedAt": datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M UTC"),
        "summary": {"models": len(model_list)},
        "thresholds": {
            "duplicate": rounded(duplicate_threshold),
            "similar": rounded(similar_threshold),
            "containment": rounded(containment_threshold),
        },
        "allPairs": all_pairs_payload,
        "models": models_payload,
        "modelList": model_list,
        "daxPool": dax_pool,
        "defaultCompare": {"a": default_a, "b": default_b},
        "reportDependencies": report_payload,
        "combinedWeights": security_context["combinedWeights"],
        "securityNotice": security_context["notice"],
        "mPool": m_pool,
        "scoreVersion": 3,
    }

    app_template = r"""
<script>
  (() => {
    const param = new URLSearchParams(window.location.search).get("scoutTheme");
    const theme =
      param || (window.matchMedia("(prefers-color-scheme: dark)").matches ? "dark" : "light");
    document.documentElement.setAttribute("data-theme", theme);
  })();
</script>
<style>
:root {
  color-scheme: light;
  --cp-bg: #f3f6f6;
  --cp-bg-elevated: #e9efef;
  --cp-surface: #ffffff;
  --cp-surface-soft: #edf2f1;
  --cp-border: #c7d3d1;
  --cp-border-strong: #687c78;
  --cp-text: #192423;
  --cp-text-muted: #4b5e5a;
  --cp-text-soft: #50625e;
  --cp-accent: #117865;
  --cp-accent-hover: #0e6961;
  --cp-accent-soft: #e0f2ed;
  --cp-accent-fg: #ffffff;
  --cp-success: #127345;
  --cp-danger: #b32931;
  --cp-warning: #8b5600;
  --cp-link: #155e9a;
  --cp-row-hover: #eaf3f1;
  --cp-row-selected: #d9ede7;
  --cp-shadow: 0 18px 48px rgba(0, 0, 0, 0.12);
  --cp-overlay: rgba(255, 255, 255, 0.8);
  --cp-panel: rgba(255, 255, 255, 0.86);
  --cp-panel-strong: rgba(255, 255, 255, 0.96);
  --cp-sheen: rgba(255, 255, 255, 0.55);
  --cp-highlight: #d9ede7;
}
html[data-theme="dark"] {
  color-scheme: dark;
  --cp-bg: #191e20;
  --cp-bg-elevated: #202729;
  --cp-surface: #232b2d;
  --cp-surface-soft: #293435;
  --cp-border: #435957;
  --cp-border-strong: #8ba29d;
  --cp-text: #eef5f3;
  --cp-text-muted: #b3c4bf;
  --cp-text-soft: #b9ccc6;
  --cp-accent: #55ddb9;
  --cp-accent-hover: #60e9d0;
  --cp-accent-soft: #24413b;
  --cp-accent-fg: #102a23;
  --cp-success: #8fd0a4;
  --cp-danger: #ffa1a7;
  --cp-warning: #f6c66f;
  --cp-link: #a1caff;
  --cp-row-hover: #293b3a;
  --cp-row-selected: #294440;
  --cp-shadow: 0 18px 48px rgba(0, 0, 0, 0.32);
  --cp-overlay: rgba(25, 30, 32, 0.88);
  --cp-panel: rgba(35, 43, 45, 0.72);
  --cp-panel-strong: rgba(35, 43, 45, 0.96);
  --cp-sheen: rgba(255, 255, 255, 0.04);
  --cp-highlight: #294440;
}
#sms-app {
  background: var(--cp-bg);
  color: var(--cp-text);
  font-family: "Segoe UI", Aptos, Calibri, -apple-system, BlinkMacSystemFont, sans-serif;
  max-width: 1600px;
  margin: 0 auto;
  font-size: 13px;
  min-height: 0;
}
#sms-app * { box-sizing: border-box; letter-spacing: 0; }
#sms-app button, #sms-app input, #sms-app select { font: inherit; max-width: 100%; }
#sms-app button { overflow-wrap: anywhere; }
#sms-app button, #sms-app select, #sms-app input[type="text"], #sms-app input[type="number"] { border-radius: 0.625rem; }
#sms-app button:focus-visible, #sms-app input:focus-visible, #sms-app select:focus-visible, #sms-app a:focus-visible, #sms-app summary:focus-visible, #sms-app [tabindex]:focus-visible {
  outline: 3px solid var(--cp-accent);
  outline-offset: 2px;
}
#sms-app .app-head { display: flex; justify-content: space-between; align-items: center; gap: 12px; padding: 8px 20px; }
#sms-app .app-head > div { min-width: 0; }
#sms-app .eyebrow { color: var(--cp-text-muted); font-size: 12px; font-weight: 700; text-transform: uppercase; }
#sms-app h1 { font-size: 20px; line-height: 1.25; margin: 0; overflow-wrap: anywhere; }
#sms-app .subtitle, #sms-app .muted { color: var(--cp-text-muted); }
#sms-app .subtitle { font-size: 13px; }
#sms-app .head-controls { display: flex; flex-direction: column; align-items: flex-end; gap: 8px; }
#sms-app .control-row, #sms-app .toolbar, #sms-app .filter-row, #sms-app .compare-controls, #sms-app .legend { display: flex; align-items: center; gap: 8px; flex-wrap: wrap; }
#sms-app .generated { color: var(--cp-text-muted); font-size: 11px; }
#sms-app .button, #sms-app .tab, #sms-app .filter, #sms-app .theme-button, #sms-app .disclosure {
  border: 1px solid var(--cp-border);
  background: var(--cp-surface);
  color: var(--cp-text);
  cursor: pointer;
  padding: 8px 12px;
}
#sms-app .button:hover, #sms-app .tab:hover, #sms-app .filter:hover, #sms-app .theme-button:hover, #sms-app .disclosure:hover { border-color: var(--cp-border-strong); }
#sms-app .button.primary { background: var(--cp-accent); border-color: var(--cp-accent); color: var(--cp-accent-fg); font-weight: 700; }
#sms-app .button.primary:hover { background: var(--cp-accent-hover); }
#sms-app .theme-button.active, #sms-app .filter.active { background: var(--cp-accent-soft); border-color: var(--cp-accent); color: var(--cp-accent); font-weight: 700; }
#sms-app .tabs { display: flex; gap: 4px; padding: 0; margin: 0 20px; border-bottom: 1px solid var(--cp-border); overflow-x: auto; }
#sms-app .tab { flex: 0 0 auto; border-color: var(--cp-surface-soft); background: var(--cp-surface-soft); font-weight: 700; }
#sms-app .tab[aria-selected="true"] { background: var(--cp-surface); border-color: var(--cp-surface); border-bottom-color: var(--cp-accent); color: var(--cp-accent); }
#sms-app .settings { margin: 8px 28px 0; }
#sms-app .settings-panel { background: var(--cp-surface); border: 1px solid var(--cp-border); border-radius: 16px; padding: 16px; box-shadow: 0 0 2px var(--cp-border), 0 1px 2px var(--cp-border); }
#sms-app .settings-grid { display: grid; gap: 12px; }
#sms-app .setting { display: grid; grid-template-columns: minmax(180px, 1fr) 2fr 84px; gap: 12px; align-items: center; }
#sms-app .setting small { display: block; color: var(--cp-text-muted); }
#sms-app input[type="range"] { width: 100%; accent-color: var(--cp-accent); }
#sms-app input[type="number"], #sms-app input[type="text"], #sms-app select { background: var(--cp-surface); border: 1px solid var(--cp-border); color: var(--cp-text); padding: 8px 10px; }
#sms-app .settings-actions { display: flex; justify-content: space-between; align-items: center; gap: 12px; margin-top: 12px; padding-top: 12px; border-top: 1px solid var(--cp-border); }
#sms-app .view { padding: 8px 20px 24px; min-width: 0; }
#sms-app .view h2 { font-size: 19px; margin: 0 0 6px; }
#sms-app .intro { color: var(--cp-text-muted); font-size: 13px; margin: 0 0 18px; line-height: 1.5; }
#sms-app .stats { display: grid; grid-template-columns: repeat(3, minmax(0, 1fr)); gap: 0; margin-bottom: 8px; border-bottom: 1px solid var(--cp-border); }
#sms-app .card, #sms-app .stat, #sms-app .empty, #sms-app .compare-summary, #sms-app .map-wrap {
  background: var(--cp-surface);
  border: 1px solid var(--cp-border);
  border-radius: 16px;
  box-shadow: 0 0 2px var(--cp-border), 0 1px 2px var(--cp-border);
}
#sms-app .stat { padding: 4px 16px 8px 0; border: 0; border-radius: 0; box-shadow: none; background: none; }
#sms-app .stat strong { display: block; font-size: 26px; line-height: 1.1; font-variant-numeric: tabular-nums; }
#sms-app .stat span { color: var(--cp-text-muted); font-size: 12px; }
#sms-app .toolbar { justify-content: space-between; margin: 16px 0 12px; }
#sms-app .search, #sms-app .report-search { flex: 1 1 220px; min-width: 0; }
#sms-app .filter { font-size: 12px; font-weight: 700; }
#sms-app .queue { display: block; }
#sms-app .candidate { padding: 16px; }
#sms-app .candidate-main { display: grid; grid-template-columns: minmax(0, 1fr) minmax(160px, 220px); gap: 24px; align-items: start; }
#sms-app .relationship { color: var(--cp-accent); font-size: 12px; font-weight: 800; text-transform: uppercase; }
#sms-app .finding { font-size: 16px; line-height: 1.4; margin: 6px 0 12px; overflow-wrap: anywhere; }
#sms-app .identity-row { display: flex; align-items: stretch; gap: 10px; margin: 8px 0; }
#sms-app .identity { flex: 1 1 0; min-width: 0; }
#sms-app .model-name { font-weight: 700; overflow-wrap: anywhere; }
#sms-app .workspace { color: var(--cp-text-muted); font-size: 12px; overflow-wrap: anywhere; }
#sms-app .relation-word { align-self: center; color: var(--cp-text-muted); font-size: 12px; font-weight: 700; }
#sms-app .plain { color: var(--cp-text-soft); font-size: 13px; line-height: 1.5; overflow-wrap: anywhere; }
#sms-app .metadata { color: var(--cp-text-muted); font-size: 11px; margin-top: 6px; overflow-wrap: anywhere; }
#sms-app .score { text-align: right; min-width: 0; overflow-wrap: anywhere; }
#sms-app .score strong { display: block; font-size: 24px; font-variant-numeric: tabular-nums; }
#sms-app .score span { display: block; color: var(--cp-text-muted); font-size: 12px; }
#sms-app .score .direction-score + .direction-score { margin-top: 12px; }
#sms-app .secondary-score { margin-top: 12px; }
#sms-app .secondary-score strong { display: block; color: var(--cp-text); font-size: 13px; }
#sms-app .caution { margin: 12px 0 0; color: var(--cp-text-soft); font-size: 12px; line-height: 1.5; }
#sms-app .candidate-actions { display: flex; gap: 8px; justify-content: flex-end; flex-wrap: wrap; margin-top: 12px; }
#sms-app .evidence { border-top: 1px solid var(--cp-border); margin-top: 12px; padding-top: 12px; }
#sms-app .score-section + .score-section { border-top: 1px solid var(--cp-border); padding-top: 16px; margin-top: 16px; }
#sms-app .score-section h4 { font-size: 14px; margin: 0 0 6px; }
#sms-app .score-section p { margin: 6px 0; }
#sms-app .coverage-list { display: block; width: 100%; max-width: none; margin: 8px 0; padding: 0; }
#sms-app .coverage-list > div { display: grid; grid-template-columns: minmax(0, 1fr) max-content; align-items: baseline; gap: 16px; padding: 8px 0; font-size: 12px; line-height: 1.5; }
#sms-app .coverage-list dt, #sms-app .coverage-list dd { display: block; position: static; float: none; clear: none; width: auto; max-width: none; min-width: 0; margin: 0; padding: 0; }
#sms-app .coverage-list dt { font-weight: 400; text-align: left; overflow-wrap: anywhere; }
#sms-app .coverage-list dd { justify-self: end; text-align: right; white-space: nowrap; font-weight: 700; font-variant-numeric: tabular-nums; }
#sms-app .evidence-grid { display: grid; grid-template-columns: repeat(auto-fit, minmax(140px, 1fr)); grid-auto-rows: 1fr; gap: 8px; width: 100%; max-width: none; margin: 12px 0; padding: 0; }
#sms-app .evidence-item { display: flex; flex-direction: column; min-width: 0; min-height: 86px; background: var(--cp-surface-soft); border-radius: 8px; padding: 10px; overflow-wrap: anywhere; }
#sms-app .evidence-item dt, #sms-app .evidence-item dd { display: block; position: static; float: none; clear: none; width: auto; max-width: none; min-width: 0; margin: 0; padding: 0; text-align: left; }
#sms-app .evidence-item dt { flex: 1; font-size: 11px; font-weight: 400; line-height: 1.4; }
#sms-app .evidence-item dd { margin-top: 4px; }
#sms-app .metric-value { display: block; font-size: 14px; line-height: 1.3; font-weight: 700; font-variant-numeric: tabular-nums; white-space: nowrap; }
#sms-app .metric-meter { display: block; width: 100%; height: 6px; margin-top: 6px; background: var(--cp-border); border-radius: 3px; overflow: hidden; }
#sms-app .metric-fill { display: block; height: 100%; background: var(--cp-accent); border-radius: inherit; }
#sms-app .metric-meter.unavailable { background: var(--cp-surface); border: 1px dashed var(--cp-border-strong); }
#sms-app .metric-scale { display: flex; justify-content: space-between; margin-top: 4px; color: var(--cp-text-muted); font-size: 9px; line-height: 1.2; font-weight: 400; }
#sms-app .score-triplet { grid-template-columns: repeat(3, minmax(0, 1fr)); }
#sms-app .score-triplet .metric-value { font-size: 20px; }
#sms-app .score-triplet .evidence-item:last-child { box-shadow: inset 0 2px 0 var(--cp-accent); }
#sms-app .metric-value.no-value { font-size: 12px; white-space: normal; }
#sms-app .security-role { border-top: 1px solid var(--cp-border); padding-top: 12px; margin-top: 12px; min-width: 0; }
#sms-app .security-role h4 { font-size: 13px; margin: 0 0 8px; overflow-wrap: anywhere; }
#sms-app .security-role .dax { margin-left: 0; font-size: 12px; }
#sms-app .security-detail { min-width: 0; flex: 1; overflow-wrap: anywhere; }
#sms-app .empty { padding: 28px; text-align: center; color: var(--cp-text-muted); }
#sms-app .method { margin-top: 8px; font-size: 12px; color: var(--cp-text-soft); line-height: 1.5; }
#sms-app .method summary { cursor: pointer; font-weight: 600; }
#sms-app .groups { display: grid; gap: 12px; }
#sms-app .group { padding: 16px; }
#sms-app .group-head { display: flex; justify-content: space-between; gap: 12px; align-items: baseline; flex-wrap: wrap; }
#sms-app .group-head h3 { margin: 0; font-size: 16px; overflow-wrap: anywhere; }
#sms-app .group-members { display: grid; gap: 4px; list-style: none; padding: 0; margin: 12px 0; }
#sms-app .group-members li { display: grid; grid-template-columns: minmax(0, 1fr) minmax(0, 1fr); gap: 8px; border-top: 1px solid var(--cp-border); padding-top: 6px; }
#sms-app .group-pickers { display: grid; grid-template-columns: minmax(0, 1fr) minmax(0, 1fr) auto; gap: 8px; align-items: end; }
#sms-app label { color: var(--cp-text-soft); font-size: 12px; font-weight: 600; min-width: 0; }
#sms-app label select { display: block; width: 100%; margin-top: 4px; }
#sms-app .compare-controls { margin-bottom: 14px; }
#sms-app .compare-controls label { flex: 1 1 230px; min-width: 0; }
#sms-app select { max-width: 100%; }
#sms-app .compare-summary { padding: 0 0 12px; margin-bottom: 12px; border: 0; border-bottom: 1px solid var(--cp-border); border-radius: 0; box-shadow: none; background: none; }
#sms-app .summary-grid { display: grid; grid-template-columns: repeat(3, minmax(0, 1fr)); gap: 16px; margin-top: 12px; }
#sms-app .summary-item { min-width: 0; overflow-wrap: anywhere; }
#sms-app .summary-item > span { display: block; color: var(--cp-text-muted); font-size: 12px; }
#sms-app .summary-item > strong { font-size: 16px; }
#sms-app .diff-section { margin-bottom: 10px; }
#sms-app .section-button { width: 100%; display: flex; justify-content: space-between; align-items: center; gap: 12px; text-align: left; background: var(--cp-surface); border: 1px solid var(--cp-border); color: var(--cp-text); border-radius: 16px; padding: 13px 16px; cursor: pointer; font-weight: 700; }
#sms-app .section-button > span { min-width: 0; overflow-wrap: anywhere; }
#sms-app .section-body { background: var(--cp-surface); border: 1px solid var(--cp-border); border-top: 0; padding: 12px 16px; border-radius: 0 0 16px 16px; }
#sms-app .diff-row { display: flex; gap: 8px; padding: 5px 7px; border-radius: 0.625rem; font-size: 12px; }
#sms-app .diff-row > span { min-width: 0; overflow-wrap: anywhere; }
#sms-app .diff-row.onlyA, #sms-app .diff-row.onlyB, #sms-app .diff-row.changed { background: var(--cp-accent-soft); }
#sms-app .status { color: var(--cp-accent); flex: 0 0 40%; font-weight: 700; }
#sms-app .dax { font-family: Consolas, "Courier New", Courier, monospace; white-space: pre-wrap; overflow-wrap: anywhere; background: var(--cp-surface-soft); border: 1px solid var(--cp-border); border-radius: 0.625rem; padding: 8px; margin: 4px 0 10px 100px; }
#sms-app .map-note { border-left: 4px solid var(--cp-warning); padding: 8px 12px; background: var(--cp-surface-soft); margin-bottom: 12px; font-size: 12px; }
#sms-app .map-wrap { overflow-x: auto; padding: 16px; }
#sms-app .matrix { border-collapse: separate; border-spacing: 3px; min-width: max-content; }
#sms-app .matrix th { max-width: 150px; color: var(--cp-text-muted); font-size: 10px; font-weight: 700; text-align: left; overflow-wrap: anywhere; }
#sms-app .matrix th.col { writing-mode: vertical-rl; transform: rotate(180deg); height: 180px; text-align: right; }
#sms-app .matrix tbody tr { background: var(--cp-surface); }
#sms-app .matrix tbody tr:nth-child(even) { background: var(--cp-surface-soft); }
#sms-app .matrix tbody td, #sms-app .matrix tbody th { background: inherit; }
#sms-app .matrix-cell { width: 44px; height: 36px; padding: 0; border: 1px solid var(--cp-border); border-radius: 0.625rem; font-size: 10px; font-weight: 800; }
#sms-app button.matrix-cell { cursor: pointer; }
#sms-app button.matrix-cell:hover { border-color: var(--cp-accent); outline: 0; box-shadow: 0 0 0 2px var(--cp-surface), 0 0 0 5px var(--cp-accent); position: relative; z-index: 1; }
#sms-app button.matrix-cell:focus-visible { border-color: var(--cp-accent); outline: 0; box-shadow: 0 0 0 2px var(--cp-surface), 0 0 0 5px var(--cp-accent); position: relative; z-index: 1; }
#sms-app .matrix-cell.duplicate { background: var(--cp-accent); color: var(--cp-accent-fg); }
#sms-app .matrix-cell.high { background: var(--cp-highlight); color: var(--cp-text); }
#sms-app .matrix-cell.low, #sms-app .matrix-cell.unavailable { background: var(--cp-surface-soft); color: var(--cp-text-muted); }
#sms-app .matrix-cell.unscored { background: var(--cp-bg-elevated); color: var(--cp-text-muted); }
#sms-app .matrix-cell.diagonal { display: inline-flex; justify-content: center; align-items: center; background: var(--cp-border); color: var(--cp-text); }
#sms-app .legend { margin-top: 12px; font-size: 11px; color: var(--cp-text-muted); }
#sms-app .legend-key { display: inline-block; width: 12px; height: 12px; border: 1px solid var(--cp-border); border-radius: 4px; vertical-align: middle; }
#sms-app .legend-key.duplicate { background: var(--cp-accent); }
#sms-app .legend-key.high { background: var(--cp-highlight); }
#sms-app .legend-key.low { background: var(--cp-surface-soft); }
#sms-app .legend-key.unscored { background: var(--cp-bg-elevated); }
#sms-app .legend-key.unavailable, #sms-app .matrix-cell.unavailable { border-style: dashed; }
#sms-app .report-count { color: var(--cp-text-soft); font-size: 11px; margin-top: 4px; }
#sms-app .report-banner { padding: 10px 12px; margin: 12px 0; border-left: 4px solid var(--cp-warning); background: var(--cp-surface-soft); font-size: 12px; overflow-wrap: anywhere; }
#sms-app .report-banner.complete { border-color: var(--cp-success); }
#sms-app .report-model { border-top: 1px solid var(--cp-border); padding: 14px 0; }
#sms-app .report-model summary { display: flex; flex-wrap: wrap; gap: 8px; align-items: baseline; cursor: pointer; }
#sms-app .report-model summary .report-count { margin-left: auto; }
#sms-app .report-list { list-style: none; margin: 8px 0 0; padding: 0; }
#sms-app .report-list li { display: grid; grid-template-columns: minmax(0, 1fr) minmax(100px, .6fr) auto; gap: 10px; align-items: baseline; padding: 9px 0; border-top: 1px solid var(--cp-border); font-size: 12px; overflow-wrap: anywhere; }
#sms-app .report-list a { color: var(--cp-link); text-underline-offset: 3px; overflow-wrap: anywhere; }
#sms-app .report-list .report-kind { color: var(--cp-text-muted); font-size: 11px; }
#sms-app .report-compare { display: grid; grid-template-columns: repeat(2, minmax(0, 1fr)); gap: 20px; }
#sms-app .report-compare h3 { font-size: 14px; margin: 0 0 6px; overflow-wrap: anywhere; }
#sms-app .report-compare .report-list li { grid-template-columns: 1fr; gap: 4px; }
#sms-app .scan-details { border-top: 1px solid var(--cp-border); padding-top: 12px; margin-top: 12px; font-size: 12px; }
#sms-app .scan-details summary { cursor: pointer; }
#sms-app .scan-row { display: grid; grid-template-columns: minmax(0, 1fr) auto auto; gap: 12px; padding: 7px 0; border-bottom: 1px solid var(--cp-border); overflow-wrap: anywhere; }
#sms-app .foot { color: var(--cp-text-muted); font-size: 11px; padding: 0 28px 22px; line-height: 1.5; }
#sms-app [hidden] { display: none !important; }
#sms-app .icon-button { display: inline-flex; align-items: center; justify-content: center; width: 34px; height: 34px; padding: 0; flex: 0 0 34px; }
#sms-app .status-strip { display: flex; gap: 16px; flex-wrap: wrap; align-items: center; min-height: 28px; margin-bottom: 8px; color: var(--cp-text-muted); font-size: 12px; }
#sms-app .status-link { color: inherit; padding: 2px 0; border: 0; background: none; text-align: left; cursor: pointer; text-decoration: underline; text-underline-offset: 3px; }
#sms-app .status-link.warning { color: var(--cp-text); border-left: 3px solid var(--cp-warning); border-radius: 0; padding-left: 7px; }
#sms-app .navigation { display: flex; align-items: center; gap: 16px; margin: 0 20px; border-bottom: 1px solid var(--cp-border); }
#sms-app .navigation .tabs { flex: 1; min-width: 0; margin: 0; border: 0; }
#sms-app .scope-toggle { display: flex; align-items: center; gap: 8px; min-height: 34px; flex-shrink: 0; }
#sms-app .scope-toggle input { margin: 0; accent-color: var(--cp-accent); }
#sms-app .scope-note { font-size: 12px; color: var(--cp-text-muted); }
#sms-app .review-filters { display: grid; grid-template-columns: minmax(160px, 1.6fr) repeat(3, minmax(115px, 1fr)); gap: 8px; align-items: end; margin: 8px 0; }
#sms-app .review-filters input, #sms-app .review-filters select { display: block; width: 100%; margin-top: 3px; padding: 6px 8px; min-height: 34px; }
#sms-app .table-summary, #sms-app .pager { display: flex; align-items: center; justify-content: space-between; flex-wrap: wrap; gap: 8px; margin: 6px 0; color: var(--cp-text-muted); font-size: 12px; }
#sms-app .table-summary h2 { font-size: 14px; margin: 0; }
#sms-app .table-scroll { width: 100%; max-width: 100%; overflow: auto; border-top: 1px solid var(--cp-border); border-bottom: 1px solid var(--cp-border); }
#sms-app .pair-table { width: 100%; min-width: 860px; border-collapse: collapse; table-layout: fixed; font-size: 13px; }
#sms-app .pair-table th { padding: 8px 10px; background: var(--cp-surface-soft); color: var(--cp-text-muted); font-size: 12px; text-align: left; font-weight: 600; }
#sms-app .pair-table td { text-align: left; padding: 9px 10px; border-top: 1px solid var(--cp-border); vertical-align: middle; overflow-wrap: anywhere; background: var(--cp-surface); color: var(--cp-text); }
#sms-app .pair-table .identity-col { width: 23%; }
#sms-app .pair-table .score-col { width: 11%; }
#sms-app .pair-table .finding-col { width: 17%; }
#sms-app .pair-table .action-col { width: 56px; }
#sms-app .pair-table tr[data-pair-key] { cursor: pointer; }
#sms-app .pair-disclosure { display: inline-flex; align-items: baseline; gap: 6px; padding: 0; margin: 0; border: 0; border-radius: 0; background: none; color: inherit; text-align: left; cursor: pointer; font-weight: inherit; }
#sms-app .pair-disclosure > span:first-child { min-width: 0; overflow-wrap: anywhere; }
#sms-app .pair-disclosure .row-chevron { flex: 0 0 12px; width: 12px; text-align: center; }
#sms-app .pair-disclosure[aria-expanded="true"] .row-chevron { transform: rotate(90deg); }
#sms-app .pair-table .pair-details > td { padding: 12px 16px 16px; background: var(--cp-surface-soft); }
#sms-app .pair-indicators .evidence-grid { margin: 0; }
#sms-app .pair-indicators .evidence-item { min-height: 0; padding: 0; background: none; }
#sms-app .pair-indicators .metric-value { font-size: 18px; }
#sms-app .pair-indicators .metric-scale { display: none; }
#sms-app .pair-table tr, #sms-app .group-table tr { background: var(--cp-surface); color: var(--cp-text); }
#sms-app .pair-table tr[data-pair-key]:hover td, #sms-app .pair-table tr:focus-within td, #sms-app .group-table tbody tr:hover td, #sms-app .group-table tr:focus-within td { background: var(--cp-row-hover); color: var(--cp-text); }
#sms-app .table-sort { display: inline-flex; align-items: center; justify-content: flex-start; text-align: left; gap: 6px; padding: 0; border: 0; border-radius: 0; background: none; color: inherit; cursor: pointer; font-weight: inherit; }
#sms-app .table-sort.active { color: var(--cp-accent); }
#sms-app .cell-value { display: block; font-variant-numeric: tabular-nums; font-weight: 600; white-space: nowrap; }
#sms-app .cell-value.unavailable { white-space: normal; font-size: 12px; color: var(--cp-text-muted); font-weight: 400; }
#sms-app .cell-bar { width: 72px; max-width: 100%; height: 3px; margin-top: 5px; background: var(--cp-border); }
#sms-app .cell-bar > span { display: block; height: 100%; background: var(--cp-accent); }
#sms-app .security-tag { display: inline-flex; align-items: center; gap: 5px; margin-top: 3px; color: var(--cp-text-muted); font-size: 12px; }
#sms-app .security-tag.warning { color: var(--cp-text); border-left: 3px solid var(--cp-warning); padding-left: 6px; }
#sms-app .finding-label { font-size: 12px; font-weight: 600; color: var(--cp-accent); }
#sms-app .pager select { padding: 4px 8px; margin-left: 6px; }
#sms-app .pager .control-row { flex-wrap: nowrap; }
#sms-app .pager button:disabled { opacity: .45; cursor: default; }
#sms-app .compare-title { display: flex; align-items: center; gap: 10px; margin-bottom: 12px; }
#sms-app .compare-title h2 { margin: 0; }
#sms-app .compare-controls { display: grid; grid-template-columns: minmax(0, 1fr) 34px minmax(0, 1fr); gap: 12px; align-items: end; margin: 0 0 10px; }
#sms-app .compare-controls select { min-height: 36px; }
#sms-app .compare-controls .model-name { margin-top: 6px; font-size: 13px; }
#sms-app .compare-flags { display: flex; align-items: center; gap: 16px; flex-wrap: wrap; margin: 8px 0; }
#sms-app .compare-flags .security-tag { margin: 0; }
#sms-app .compare-flags label { margin-left: auto; }
#sms-app .score-triplet { margin: 12px 0; }
#sms-app .score-triplet .evidence-item { padding: 8px 16px 8px 0; border-radius: 0; background: none; min-height: 60px; }
#sms-app .score-triplet .evidence-item dt { font-size: 12px; }
#sms-app .score-triplet .evidence-item:last-child { box-shadow: none; }
#sms-app .score-triplet .evidence-item:last-child .metric-value { color: var(--cp-accent); }
#sms-app .score-triplet .metric-scale { display: none; }
#sms-app .comparison-context { display: grid; grid-template-columns: 2fr 1fr; align-items: center; gap: 20px; }
#sms-app .comparison-context .coverage-list { margin: 0; }
#sms-app .comparison-context .coverage-list > div { padding: 3px 0; }
#sms-app .comparison-context .schema-counts { font-size: 12px; text-align: right; color: var(--cp-text-muted); }
#sms-app .section-button { border-radius: 4px; padding: 9px 12px; font-size: 13px; }
#sms-app .section-body { border-radius: 0 0 4px 4px; padding: 8px 12px; }
#sms-app .formula-detail { font-size: 12px; margin: 4px 0 8px; }
#sms-app .formula-detail summary { cursor: pointer; color: var(--cp-link); }
#sms-app .rule-row { border-top: 1px solid var(--cp-border); padding: 8px 0; }
#sms-app .rule-row summary { display: flex; align-items: baseline; gap: 10px; cursor: pointer; }
#sms-app .rule-row .status { flex-basis: 34%; min-width: 0; overflow-wrap: anywhere; }
#sms-app .report-scope { display: flex; flex-wrap: wrap; gap: 8px 16px; font-size: 12px; margin-bottom: 12px; color: var(--cp-text-muted); }
#sms-app .report-scope .warning { color: var(--cp-text); border-left: 3px solid var(--cp-warning); padding-left: 6px; }
#sms-app .help-panel { position: fixed; inset: auto; margin: 0; color: var(--cp-text); background: var(--cp-surface); border: 1px solid var(--cp-border); border-radius: 8px; max-width: 620px; width: calc(100% - 32px); max-height: 560px; padding: 20px; overflow: auto; overscroll-behavior: contain; font-size: 13px; line-height: 1.55; }
#sms-app .help-panel::backdrop { background: var(--cp-overlay); }
#sms-app .help-head { position: sticky; top: 0; z-index: 1; background: var(--cp-surface); display: flex; align-items: center; justify-content: space-between; gap: 12px; }
#sms-app .help-panel h2 { font-size: 18px; margin: 0; }
#sms-app .help-panel h3 { font-size: 14px; margin: 20px 0 6px; }
#sms-app .help-panel p { margin: 8px 0; }
#sms-app .help-panel .help-facts { display: grid; gap: 6px; }
#sms-app .help-facts div { display: grid; grid-template-columns: 130px minmax(0, 1fr); gap: 12px; overflow-wrap: anywhere; }
#sms-app .help-facts strong { font-weight: 600; }
#sms-app .explorer-toolbar { display: flex; gap: 12px; align-items: end; flex-wrap: wrap; margin: 10px 0 12px; }
#sms-app .explorer-toolbar label { flex: 1 1 220px; }
#sms-app .explorer-toolbar input, #sms-app .explorer-toolbar select { display: block; width: 100%; margin-top: 3px; min-height: 34px; }
#sms-app .group-table { width: 100%; min-width: 640px; border-collapse: collapse; font-size: 13px; }
#sms-app .group-table th, #sms-app .group-table td { text-align: left; padding: 9px 12px; border-bottom: 1px solid var(--cp-border); }
#sms-app .group-table th { background: var(--cp-surface-soft); font-size: 12px; color: var(--cp-text-muted); font-weight: 600; }
#sms-app .group-table td { background: var(--cp-surface); color: var(--cp-text); font-variant-numeric: tabular-nums; }
#sms-app .group-table tr.selected td, #sms-app .group-table tr.selected:hover td, #sms-app .group-table tr.selected:focus-within td { background: var(--cp-row-selected); color: var(--cp-text); }
#sms-app .group-detail { padding: 16px 0 0; }
#sms-app .group-detail h3 { margin: 0; font-size: 15px; }
#sms-app .group-members li { display: grid; grid-template-columns: minmax(0, 2fr) minmax(0, 1.25fr) minmax(100px, 1fr) auto; padding: 8px 0; align-items: baseline; }
#sms-app .member-security, #sms-app .member-reports { font-size: 12px; color: var(--cp-text-muted); }
#sms-app .group-pickers select { min-height: 36px; }
#sms-app .matrix-container { border-top: 1px solid var(--cp-border); border-bottom: 1px solid var(--cp-border); overflow: auto; max-height: 620px; }
#sms-app .matrix { border-collapse: separate; border-spacing: 2px; margin: 0; background: var(--cp-surface); }
#sms-app .matrix thead th { position: sticky; top: 0; z-index: 2; background: var(--cp-surface); }
#sms-app .matrix th:first-child { position: sticky; left: 0; z-index: 3; min-width: 148px; background: var(--cp-surface); }
#sms-app .matrix thead th:first-child { z-index: 4; }
#sms-app .matrix th { font-size: 12px; padding: 4px; min-width: 46px; }
#sms-app .matrix th.col { writing-mode: initial; transform: none; height: 128px; width: 46px; max-width: 46px; text-align: center; vertical-align: bottom; }
#sms-app .axis-label { display: inline-block; writing-mode: vertical-rl; transform: rotate(180deg); max-height: 120px; max-width: 40px; overflow: hidden; }
#sms-app .matrix-cell { border-radius: 4px; font-size: 12px; width: 44px; height: 34px; }
#sms-app .matrix-cell.unscored { background: var(--cp-bg-elevated); }
#sms-app .matrix-cell.outside { background: var(--cp-surface-soft); color: var(--cp-text-muted); border: 1px solid var(--cp-border); cursor: default; }
#sms-app .matrix-cell.diagonal { font-weight: 400; }
#sms-app .map-toolbar h2 { margin: 0; font-size: 16px; }
#sms-app .map-detail { min-height: 30px; padding: 6px 0; color: var(--cp-text-muted); font-size: 12px; overflow-wrap: anywhere; }
#sms-app .legend { font-size: 12px; margin: 8px 0; }
#sms-app .security-status-pair { display: flex; flex-wrap: wrap; gap: 6px 12px; }
@media (max-width: 720px) {
  #sms-app .app-head { flex-direction: column; padding: 20px 16px 12px; }
  #sms-app .head-controls { align-items: flex-start; }
  #sms-app .tabs, #sms-app .settings { margin-left: 16px; margin-right: 16px; }
  #sms-app .view { padding: 18px 16px 22px; }
  #sms-app .stats, #sms-app .summary-grid, #sms-app .report-compare { grid-template-columns: 1fr; }
  #sms-app .candidate-main { grid-template-columns: 1fr; gap: 12px; }
#sms-app .score { text-align: left; }
  #sms-app .identity-row { flex-direction: column; }
  #sms-app .relation-word { align-self: flex-start; }
  #sms-app .setting, #sms-app .group-pickers { grid-template-columns: 1fr; }
  #sms-app .group-members li, #sms-app .report-list li, #sms-app .scan-row { grid-template-columns: 1fr; gap: 4px; }
  #sms-app .report-model summary { flex-direction: column; gap: 4px; }
  #sms-app .report-model summary .report-count { margin-left: 0; }
  #sms-app .candidate-actions, #sms-app .settings-actions { flex-wrap: wrap; justify-content: flex-start; }
  #sms-app .evidence-grid { grid-template-columns: repeat(2, minmax(0, 1fr)); }
  #sms-app .score-triplet { grid-template-columns: repeat(2, minmax(0, 1fr)); }
  #sms-app .score-triplet .evidence-item:last-child { grid-column: 1 / -1; }
  #sms-app .section-button { flex-wrap: wrap; }
  #sms-app .dax { margin-left: 0; }
  #sms-app .app-head { padding: 12px; flex-direction: row; align-items: flex-start; }
  #sms-app h1 { font-size: 17px; }
  #sms-app .head-controls { align-items: flex-end; }
  #sms-app .head-controls .control-row { gap: 4px; flex-wrap: nowrap; }
  #sms-app .tabs, #sms-app .settings { margin-left: 12px; margin-right: 12px; }
  #sms-app .view { padding: 12px; }
  #sms-app .stats { grid-template-columns: repeat(3, minmax(0, 1fr)); }
  #sms-app .stat strong { font-size: 24px; }
  #sms-app .stat span { font-size: 12px; }
  #sms-app .review-filters { grid-template-columns: repeat(2, minmax(0, 1fr)); }
  #sms-app .review-filters .search-label { grid-column: 1 / -1; }
  #sms-app .navigation { margin: 0 12px; flex-wrap: wrap; gap: 0 12px; }
  #sms-app .navigation .tabs { flex-basis: 100%; margin: 0; }
  #sms-app .compare-controls { grid-template-columns: minmax(0, 1fr) 34px; }
  #sms-app .compare-controls > label:last-child { grid-column: 1 / -1; }
  #sms-app .compare-flags label { margin-left: 0; }
  #sms-app .comparison-context { grid-template-columns: minmax(0, 1fr); gap: 8px; }
  #sms-app .comparison-context .schema-counts { text-align: left; }
  #sms-app .score-triplet { grid-template-columns: repeat(3, minmax(0, 1fr)); }
  #sms-app .score-triplet .evidence-item:last-child { grid-column: auto; }
  #sms-app .score-triplet .metric-value { font-size: 18px; }
  #sms-app .score-triplet .metric-value.no-value { font-size: 12px; }
  #sms-app .group-members li { grid-template-columns: minmax(0, 1fr); gap: 4px; }
  #sms-app .group-pickers { grid-template-columns: minmax(0, 1fr); }
  #sms-app .matrix-container { max-height: 480px; }
  #sms-app .explorer-toolbar { gap: 8px; }
}
</style>
<div id="sms-app"></div>
<script>
(function(){
  var DATA = __APP_DATA__;
  var root = document.getElementById('sms-app');
  if(!root){ return; }
  var MODELS = DATA.models || {}, MLIST = DATA.modelList || [], DAXPOOL = DATA.daxPool || [];
  var REPORTS = DATA.reportDependencies || {status:'not_collected',byModel:{}};
  var DEFAULTS = DATA.thresholds || {duplicate:.95, similar:.70, containment:.95};
  var LABELS = {duplicate:'Possible duplicates',containment:'Schema coverage',overlap:'Shared structure'};
  var COVERAGE_MEANING = 'A weighted match across cataloged schema definitions, not a percentage of objects or data values. Security is excluded.';
  var SCORE_DIFFERENCE = 'Schema similarity compares both complete models. Extra content can lower this score even when one model has high coverage within the other.';
  var REVIEW_CAUTION = 'This is a review candidate, not confirmation that one model can replace the other.';
  var initialTheme = document.documentElement.getAttribute('data-theme') || 'light';
  var state = {
    tab:'review', filter:'all', search:'', theme:initialTheme, settingsOpen:false,
    thresholds:{duplicate:DEFAULTS.duplicate, similar:DEFAULTS.similar, containment:DEFAULTS.containment},
    draft:{duplicate:DEFAULTS.duplicate, similar:DEFAULTS.similar, containment:DEFAULTS.containment},
    cmpA:(DATA.defaultCompare||{}).a || '', cmpB:(DATA.defaultCompare||{}).b || '',
    cmpDiffOnly:true, openSections:{}, expandedPairs:Object.create(null), groupSelections:{}, reportSearch:'', reportsOnly:false,
    workspace:'all', security:'all', powerQuery:'all', crossOnly:false, sort:'rank', sortDirection:'desc', page:1, pageSize:25,
    helpOpen:false, returnView:null, groupSearch:'', selectedGroup:'',
    mapSearch:'', mapWorkspace:'all', mapPage:1, mapSize:40
  };
  try {
    var savedTheme=localStorage.getItem('sms-theme');
    if(savedTheme==='light'||savedTheme==='dark'){ state.theme=savedTheme; }
    var savedThresholds=JSON.parse(localStorage.getItem('sms-thresholds-v2')||'null');
    if(savedThresholds){ ['duplicate','similar','containment'].forEach(function(key){ if(typeof savedThresholds[key]==='number'&&Number.isFinite(savedThresholds[key])){ state.thresholds[key]=Math.max(0,Math.min(1,savedThresholds[key])); } }); state.draft=Object.assign({},state.thresholds); }
  } catch(e){}
  if(!state.cmpA && MLIST[0]){ state.cmpA=MLIST[0].id; }
  if(!state.cmpB && MLIST[1]){ state.cmpB=MLIST[1].id; }

  function esc(s){ return String(s==null?'':s).replace(/[&<>"']/g,function(c){ return {'&':'&amp;','<':'&lt;','>':'&gt;','"':'&quot;',"'":'&#39;'}[c]; }); }
  function validScore(value){ return typeof value==='number'&&Number.isFinite(value)&&value>=0&&value<=1; }
  function score(value){ return validScore(value)?(Math.round(value*1000)/10).toFixed(1)+'%':'Unavailable'; }
  function model(id){ return MODELS[id] || {name:id||'Model unavailable',workspace:''}; }
  function modelWorkspaceId(id){var value=model(id).workspaceId;return typeof value==='string'?value.trim().toLowerCase():'';}
  function inComparisonScope(firstId,secondId){var first=modelWorkspaceId(firstId),second=modelWorkspaceId(secondId);return !state.crossOnly||!!(first&&second&&first!==second);}
  function scopeExclusion(firstId,secondId){return modelWorkspaceId(firstId)&&modelWorkspaceId(secondId)?'Same workspace':'Workspace unknown';}
  function setComparisonScope(enabled){state.crossOnly=!!enabled;state.page=1;state.mapPage=1;}
  function clearReviewFilters(){Object.assign(state,{filter:'all',search:'',workspace:'all',security:'all',powerQuery:'all',page:1});}
  function identity(id){ var entry=model(id); return '<div class="identity"><div class="model-name">'+esc(entry.name)+'</div><div class="workspace">Workspace: '+esc(entry.workspace||'Unavailable')+'</div><div class="report-count">'+esc(reportCountText(id))+'</div></div>'; }
  function pairModelLabel(id,otherId){
    var entry=model(id),other=model(otherId),label=entry.name;
    if(String(entry.name).toLowerCase()===String(other.name).toLowerCase()){
      label+=' (Workspace: '+(entry.workspace||'Unavailable')+')';
      if(String(entry.workspace).toLowerCase()===String(other.workspace).toLowerCase()){label+=' ['+id+']';}
    }
    return label;
  }
  function pairKey(a,b){ return [String(a),String(b)].sort().join('|'); }
  var pairIndex=Object.create(null);
  (DATA.allPairs||[]).forEach(function(pair){pairIndex[pairKey(pair.idA,pair.idB)]=pair;});
  function pairMap(){ return pairIndex; }
  function daxText(item){ return item&&item.daxId>=0 ? (DAXPOOL[item.daxId]||'') : ''; }
  function mText(item){ var pool=DATA.mPool||[]; return item&&item.mId>=0 ? (pool[item.mId]||'') : ''; }
  function powerQueryDiffers(pair){ return pair.powerQueryStatus==='one_sided'||(pair.powerQueryStatus==='compared'&&validScore(pair.powerQuery)&&pair.powerQuery<1); }
  function tierAt(value){ return !validScore(value)?'unassessed':(value>=state.thresholds.duplicate?'duplicate':(value>=state.thresholds.similar?'similar':'distinct')); }
  function securityLabel(pair){return !pair?'Not scored':({match:'Matching definitions',different:'Security definitions differ',not_applicable:'Not applicable',unknown:'Security not assessed'}[pair.securityStatus]||'Security not assessed');}
  function securityScoreText(pair){return pair&&pair.securityStatus==='not_applicable'?'Not applicable':score(pair?pair.security:null);}
  function blendText(pair){
    if(!pair){return 'No saved score for this pair.';}
    if(pair.scoreMode==='schema_only_no_security'){return 'No model-level security definitions in either model: overall equals schema.';}
    if(!validScore(pair.combined)){return 'A complete security comparison is required for the overall score.';}
    var weights=DATA.combinedWeights;
    return weights?score(weights.schema)+' schema + '+score(weights.security)+' security.':'Overall weights unavailable.';
  }
  function securityNoticeHTML(pair){
    if(!pair){return '<p class="plain">Security comparison not scored.</p>';}
    if(pair.securityStatus==='different'){return '<div class="report-banner security-banner" role="status"><strong>Security definitions differ</strong><div>Duplicate labels use the overall score. Review the permission differences before considering replacement.</div></div>';}
    if(pair.securityStatus==='unknown'){return '<div class="report-banner security-banner" role="status"><strong>Security not assessed</strong><div>Missing, incomplete, or inconsistent security evidence. Run catalog and scoring again; schema scores remain separate.</div></div>';}
    if(pair.securityStatus==='not_applicable'){return '<p class="plain security-status">No model-level security definitions found in either model. Overall equals schema.</p>';}
    return '<p class="plain security-status">Matching security definitions. Role assignments and effective user access are not compared.</p>';
  }
  function scoreSummaryHTML(pair){return '<dl class="evidence-grid score-triplet">'+metricHTML('Schema',pair?pair.schema:null,pair?'Unavailable':'Not scored')+metricHTML('Security',pair?pair.security:null,pair?securityScoreText(pair):'Not scored')+metricHTML('Overall',pair?pair.combined:null,pair?'Unavailable':'Not scored')+'</dl>';}
  function coverageContext(pair){
    var directions=pair?[
      {sourceId:pair.idA,targetId:pair.idB,value:pair.aInB},
      {sourceId:pair.idB,targetId:pair.idA,value:pair.bInA}
    ]:[];
    var available=directions.filter(function(direction){return direction.sourceId&&direction.targetId&&validScore(direction.value);});
    var strongest=available.reduce(function(best,direction){return !best||direction.value>best.value?direction:best;},null);
    var matches=available.filter(function(direction){return direction.value>=state.thresholds.containment;});
    return {directions:directions,strongest:strongest,both:matches.length===2,qualifies:matches.length>0};
  }
  function directionLabel(direction){return pairModelLabel(direction.sourceId,direction.targetId)+' within '+pairModelLabel(direction.targetId,direction.sourceId);}
  function coverageRowsHTML(pair){
    if(!pair){return '<div class="plain">Not scored</div>';}
    return '<dl class="coverage-list">'+coverageContext(pair).directions.map(function(direction){return '<div><dt>Coverage score: '+esc(directionLabel(direction))+'</dt><dd>'+score(direction.value)+'</dd></div>';}).join('')+'</dl>';
  }

  function reportsAvailable(){return REPORTS.status==='complete'||REPORTS.status==='partial';}
  function reportsFor(modelId){return reportsAvailable()?((REPORTS.byModel||{})[modelId]||[]):[];}
  function reportCountText(modelId){
    if(!reportsAvailable()){return 'Report dependencies unknown';}
    var count=reportsFor(modelId).length;
    if(REPORTS.status==='partial'){
      return count?count+' known linked report'+(count===1?'':'s')+' (scan incomplete)':'No linked reports found (scan incomplete)';
    }
    return count+' linked report'+(count===1?'':'s')+' in scanned scope';
  }
  function reportStatusHTML(){
    var labels={complete:'Report scan complete within scanned scope',partial:'Partial report scan: dependencies may be missing',not_collected:'Report dependencies were not collected',unavailable:'Report dependency data is unavailable',inconsistent:'Report snapshots do not agree; dependencies are unknown'};
    var scope=REPORTS.scope||'Unknown scope';
    if(scope==='visible_workspaces'){scope='Workspaces visible to the scanning identity';}
        if(scope==='admin_workspaces'){scope='Active regular workspaces (admin discovery)';}
        if(scope.startsWith('admin_workspace_filter:')){scope='Workspace (admin discovery): '+scope.slice('admin_workspace_filter:'.length);}
    else if(scope.indexOf('workspace_filter:')===0){scope='Workspace filter: '+scope.slice('workspace_filter:'.length);}
    var detail=reportsAvailable()?'Report snapshot: '+esc(REPORTS.scannedAt||'Time unavailable')+' · '+esc(REPORTS.workspaceCount||0)+' workspaces · '+esc(scope):'Report dependencies unknown';
    return '<div class="report-banner '+(REPORTS.status==='complete'?'complete':'')+'" role="status"><strong>'+esc(labels[REPORTS.status]||'Report dependencies unknown')+'</strong><div class="metadata">'+detail+'</div>'+(reportsAvailable()?'<div class="plain">Direct report links observed in the scanned scope, not usage. Zero does not prove a model is unused.</div>':'')+'</div>';
  }
  function reportListHTML(reports){
    if(!reports.length){return '';}
    return '<ul class="report-list">'+reports.map(function(report){
      var label=esc(report.name),url;
      try{url=new URL(report.url);if(url.protocol==='https:'&&url.hostname==='app.powerbi.com'&&!url.port&&!url.username&&!url.password){label='<a href="'+esc(url.href)+'" target="_blank" rel="noopener noreferrer">'+label+'</a>';}}catch(error){}
      return '<li><span>'+label+'</span><span class="workspace">Workspace: '+esc(report.workspace)+'</span><span class="report-kind">'+esc(report.reason||(report.crossWorkspace?'Cross-workspace':'Same workspace'))+'</span></li>';
    }).join('')+'</ul>';
  }
  function reportCompareHTML(){
    var content='<div class="report-scope"><span class="'+(REPORTS.status==='complete'?'':'warning')+'">Report scan: '+shortReportStatus()+'</span><span>'+esc(reportScopeText())+'</span><button class="status-link" data-help>Scan details</button></div>'+
      '<div class="report-compare">'+[state.cmpA,state.cmpB].map(function(modelId){var count=reportsAvailable()?reportsFor(modelId).length+(REPORTS.status==='partial'?' known reports':' linked reports'):'Unknown';return '<div><h3>'+esc(model(modelId).name)+'</h3><div class="workspace">'+esc(model(modelId).workspace||'Unavailable')+'</div><div class="report-count">'+count+'</div>'+reportListHTML(reportsFor(modelId))+'</div>';}).join('')+'</div>';
    var summary=REPORTS.status==='complete'?'Observed direct links':(REPORTS.status==='partial'?'Scan incomplete':'Dependencies unknown');
    return sectionHTML('reports','Dependent reports',summary,[content]);
  }
  function reportRowsHTML(){
    var query=state.reportSearch.trim().toLowerCase();
    var filtered=MLIST.filter(function(entry){
      var reports=reportsFor(entry.id);if(state.reportsOnly&&!reports.length){return false;}
      return !query||[entry.name,entry.workspace].concat(reports.map(function(report){return report.name+' '+report.workspace;})).join(' ').toLowerCase().indexOf(query)>=0;
    });
    if(!filtered.length){return '<div class="muted">No matching models or reports in this inventory.</div>';}
    return filtered.map(function(entry){var reports=reportsFor(entry.id);return '<details class="report-model"'+(reports.length?' open':'')+'><summary><span class="model-name">'+esc(entry.name)+'</span><span class="workspace">Workspace: '+esc(entry.workspace)+'</span><span class="report-count">'+esc(reportCountText(entry.id))+'</span></summary>'+reportListHTML(reports)+'</details>';}).join('');
  }
  function reportsHTML(){
    var overview='<h2>Report dependencies</h2><p class="intro">Reports linked directly to cataloged models. These links provide impact context and do not change model scores.</p>'+reportStatusHTML();
    if(!reportsAvailable()){return overview;}
    var scans=(REPORTS.scans||[]).map(function(scan){return '<div class="scan-row"><span>'+esc(scan.workspace)+'</span><span>'+esc(scan.status)+(scan.errorType?' · '+esc(scan.errorType):'')+'</span><span>'+esc(scan.reportCount)+' reports found</span></div>';}).join('');
    return overview+'<div class="plain">'+esc(REPORTS.linkedReportCount||0)+' direct links to cataloged models · '+esc(REPORTS.reportCount||0)+' reports found'+(REPORTS.status==='partial'?' (scan incomplete)':' in scanned scope')+'</div><div class="toolbar"><input class="report-search" type="text" aria-label="Search report dependencies" placeholder="Search models, reports or workspaces" value="'+esc(state.reportSearch)+'"><label><input type="checkbox" data-reports-only'+(state.reportsOnly?' checked':'')+'> Models with linked reports found</label></div><div class="report-models">'+reportRowsHTML()+'</div>'+((REPORTS.otherReports||[]).length?'<h3>Reports without a resolved catalog link</h3>'+reportListHTML(REPORTS.otherReports):'')+'<details class="scan-details"><summary>Workspace scan status</summary>'+scans+'</details>';
  }

  function classify(pair){
    var tier=tierAt(pair.combined),context=coverageContext(pair);
    var coverage=context.strongest?context.strongest.value:(validScore(pair.containedCoverage)?pair.containedCoverage:pair.containment);
    if(tier==='duplicate'){ return {kind:'duplicate',label:LABELS.duplicate,rank:0,primary:pair.combined}; }
    if(validScore(coverage)&&coverage>=state.thresholds.containment){ return {kind:'containment',label:LABELS.containment,rank:1,primary:coverage}; }
    if(tier==='similar'){ return {kind:'overlap',label:LABELS.overlap,rank:2,primary:pair.combined}; }
    return null;
  }
  function candidates(){
    return (DATA.allPairs||[]).filter(function(pair){return inComparisonScope(pair.idA,pair.idB);}).map(function(p){ var c=classify(p); return c?{pair:p,category:c}:null; }).filter(Boolean).sort(function(a,b){ return a.category.rank-b.category.rank || b.category.primary-a.category.primary; });
  }
  function duplicatePairs(){ return (DATA.allPairs||[]).filter(function(pair){ return inComparisonScope(pair.idA,pair.idB)&&tierAt(pair.combined)==='duplicate'; }); }
  function buildGroups(){
    var dup=duplicatePairs(), parent={};
    function find(x){ if(parent[x]!==x){ parent[x]=find(parent[x]); } return parent[x]; }
    function add(x){ if(!(x in parent)){ parent[x]=x; } }
    function join(a,b){ add(a);add(b);var ra=find(a),rb=find(b);if(ra!==rb){parent[rb]=ra;} }
    dup.forEach(function(p){join(p.idA,p.idB);});
    var grouped={}; Object.keys(parent).forEach(function(id){var r=find(id);(grouped[r]=grouped[r]||[]).push(id);});
    var pmap=pairMap(), groups=[];
    Object.keys(grouped).forEach(function(key){
      var members=grouped[key]; if(members.length<2){return;}
      var strongest=null;
      for(var i=0;i<members.length;i++){for(var j=i+1;j<members.length;j++){var p=pmap[pairKey(members[i],members[j])];if(p&&inComparisonScope(p.idA,p.idB)&&tierAt(p.combined)==='duplicate'&&(!strongest||p.combined>strongest.combined)){strongest=p;}}}
      if(!strongest){return;}
      members.sort(function(a,b){return (model(a).name||'').localeCompare(model(b).name||'')||(model(a).workspace||'').localeCompare(model(b).workspace||'');});
      var fingerprints=new Set(),unknown=false;
      members.forEach(function(id){var security=model(id).security||{};if(security.status==='complete'){fingerprints.add(security.fingerprint);}else{unknown=true;}});
      groups.push({members:members,strongest:strongest,securityDifferent:fingerprints.size>1,securityUnknown:unknown});
    });
    groups.sort(function(a,b){return b.members.length-a.members.length||b.strongest.combined-a.strongest.combined;});
    return groups;
  }
  function relationshipText(pair,category){
    if(!pair){return 'This pair has no saved analysis score.';}
    if(!category){return validScore(pair.combined)?'This pair does not meet the current review thresholds.':'Overall similarity is unavailable; schema evidence remains separate.';}
    if(category.kind==='duplicate'){
      return pair.combined>=.95?'These models have very high overall similarity.':'These models meet the possible-duplicate threshold.';
    }
    if(category.kind==='containment'){
      var context=coverageContext(pair),direction=context.strongest;
      if(context.both){return 'Both models meet the schema coverage threshold.';}
      if(!direction){return 'Schema coverage was flagged, but its direction is unavailable.';}
      var source=pairModelLabel(direction.sourceId,direction.targetId),target=pairModelLabel(direction.targetId,direction.sourceId);
      return direction.value>=.95?'Almost all of '+source+' is represented in '+target+'.':source+' has a '+score(direction.value)+' schema coverage score within '+target+'.';
    }
    return 'These models meet the overall-similarity threshold, below the possible-duplicate threshold.';
  }
  function relevantSection(pair){
    if(!pair){return 'tables';}
    if(pair.securityStatus==='different'||pair.securityStatus==='unknown'){return 'security';}
    var overlap=pair.jaccard||{};
    var values=[['tables',overlap.tables],['measures',Math.min(validScore(overlap.measures)?overlap.measures:1,validScore(pair.daxCosine)?pair.daxCosine:1)],['relationships',overlap.relationships],['datasources',overlap.datasources],['queries',pair.powerQueryStatus==='not_applicable'?null:pair.powerQuery]];
    values.sort(function(first,second){return (validScore(first[1])?first[1]:1)-(validScore(second[1])?second[1]:1);}); return values[0][0];
  }

  function tabsHTML(){
    return '<div class="navigation"><div class="tabs" role="tablist" aria-label="Similarity results views">'+[
      ['review','Review'],['groups','Groups'],['map','Similarity map'],['compare','Compare']
    ].map(function(t){var selected=state.tab===t[0];return '<button class="tab" role="tab" id="tab-'+t[0]+'" aria-selected="'+selected+'" aria-controls="panel-'+t[0]+'" tabindex="'+(selected?'0':'-1')+'" data-tab="'+t[0]+'">'+t[1]+'</button>';}).join('')+'</div><label class="scope-toggle"><input type="checkbox" data-cross-only'+(state.crossOnly?' checked':'')+'>Cross-workspace only</label></div>';
  }
  function headerHTML(){
    var nextTheme=state.theme==='dark'?'light':'dark';
    return '<header class="app-head"><h1>Semantic Model Similarity</h1><div class="head-controls"><div class="control-row">'+
      '<button class="button icon-button" data-settings title="Review thresholds" aria-label="Review thresholds" aria-expanded="'+state.settingsOpen+'" aria-controls="scoring-settings"><span aria-hidden="true">&#9881;</span></button>'+
      '<button class="button icon-button" data-theme-set="'+nextTheme+'" title="Switch to '+nextTheme+' theme" aria-label="Switch to '+nextTheme+' theme"><span aria-hidden="true">'+(nextTheme==='light'?'&#9788;':'&#9790;')+'</span></button>'+
      '<button class="button icon-button" data-help title="Definitions and scan details" aria-label="Definitions and scan details" aria-haspopup="dialog"><span aria-hidden="true">?</span></button></div></div></header>';
  }
  function settingsHTML(){
    function row(key,label,hint){var value=Math.round(state.draft[key]*10000)/100;return '<div class="setting"><label for="range-'+key+'">'+label+'<small>'+hint+'</small></label><input id="range-'+key+'" type="range" min="0" max="100" step="0.01" value="'+value+'" data-draft-range="'+key+'" aria-label="'+label+' minimum score percent"><input type="number" min="0" max="100" step="0.01" required value="'+value+'" data-draft-number="'+key+'" aria-label="'+label+' minimum score percent value"></div>'; }
    return '<div class="settings" id="scoring-settings"'+(state.settingsOpen?'':' hidden')+'><div class="settings-panel"><div class="settings-grid">'+row('duplicate',LABELS.duplicate,'Overall (%)')+row('similar',LABELS.overlap,'Overall (%)')+row('containment',LABELS.containment,'Directional coverage (%)')+'</div><div id="threshold-error" class="security-tag warning" role="alert" hidden>Enter percentages from 0 to 100.</div><div class="settings-actions"><button class="status-link" data-help>Calculation details</button><div class="control-row"><button class="button" data-reset>Reset defaults</button><button class="button primary" data-apply>Apply thresholds</button></div></div></div></div>';
  }

  function securityTagHTML(pair){
    var status=pair?pair.securityStatus:'unscored';
    var labels={match:'Security matches',different:'Security differs',not_applicable:'No model roles',unknown:'Not assessed',unscored:'Not scored'};
    return '<span class="security-tag'+(status==='different'||status==='unknown'?' warning':'')+'">'+esc(labels[status]||labels.unknown)+'</span>';
  }
  function shortReportStatus(){
    return {complete:'Complete',partial:'Partial',not_collected:'Not collected',unavailable:'Unavailable',inconsistent:'Inconsistent'}[REPORTS.status]||'Unknown';
  }
  function reportScopeText(){
    var scope=REPORTS.scope||'Unknown';
    if(scope==='visible_workspaces'){return 'Visible workspaces';}
    if(scope==='admin_workspaces'){return 'Active regular workspaces (admin discovery)';}
    if(scope.startsWith('admin_workspace_filter:')){return scope.slice('admin_workspace_filter:'.length)+' (admin discovery)';}
    return scope.startsWith('workspace_filter:')?scope.slice('workspace_filter:'.length):scope;
  }
  function statusStripHTML(){
    var unknown=MLIST.filter(function(entry){return (model(entry.id).security||{}).status!=='complete';}).length;
    var unknownWorkspace=state.crossOnly?MLIST.filter(function(entry){return !modelWorkspaceId(entry.id);}).length:0;
    var securityText=unknown?unknown+' model'+(unknown===1?'':'s')+' not assessed':(DATA.securityNotice?'Analysis needs review':'Assessed');
    return '<div class="status-strip"><button class="status-link'+(unknown||DATA.securityNotice?' warning':'')+'" data-help>Security: '+esc(securityText)+'</button>'+
      '<button class="status-link'+(REPORTS.status==='complete'?'':' warning')+'" data-help>Reports: '+shortReportStatus()+'</button>'+
      '<button class="status-link" data-help>Scan details</button>'+(unknownWorkspace?'<span class="security-tag warning">Workspace unknown: '+unknownWorkspace+' model'+(unknownWorkspace===1?'':'s')+'</span>':'')+'</div>';
  }
  function helpHTML(){
    var weights=DATA.combinedWeights;
    var weightText=weights?score(weights.schema)+' schema + '+score(weights.security)+' security':'Weights unavailable';
    return '<dialog id="sms-help" class="help-panel" aria-labelledby="help-title"><div class="help-head"><h2 id="help-title">Definitions &amp; scan details</h2><button class="button icon-button" data-help-close aria-label="Close definitions" title="Close"><span aria-hidden="true">&#215;</span></button></div>'+
      '<h3>Current evidence</h3><div class="help-facts"><div><strong>View generated</strong><span>'+esc(DATA.generatedAt||'Unknown')+'</span></div><div><strong>Report snapshot</strong><span>'+esc(REPORTS.scannedAt||'Unknown')+'</span></div><div><strong>Report scope</strong><span>'+esc(reportScopeText())+'</span></div><div><strong>Report scan</strong><span>'+shortReportStatus()+'</span></div><div><strong>Overall weights</strong><span>'+esc(weightText)+'</span></div></div>'+
      (DATA.securityNotice?'<p role="status">'+esc(DATA.securityNotice)+'</p>':'')+
      '<h3>Scores</h3><p><strong>Overall</strong> drives possible-duplicate labels and groups. Security differences remain warnings, not a veto. Threshold controls reclassify saved scores; they do not rescan or recalculate them.</p>'+
      '<p><strong>Schema</strong> blends five overlap signals and local TF-IDF text similarity for DAX and Power Query (M). Overlap is shared unique definitions divided by all unique definitions. Text similarity is resemblance, not a count of matching formulas. Models without measures use structural or model text. Power Query is skipped when neither model has M and scores 0 when only one does. Sources are legacy connections plus the upstream sources M connector calls reference.</p>'+
      '<p><strong>Security</strong> compares role rules, not security strength. Roles align one-to-one independently of their names. Exact RLS filter text, table/column permissions, and applicable propagation settings are compared. Members and effective access are not assessed.</p>'+
      '<p><strong>Coverage</strong> is directional: A within B and B within A are separate weighted schema measures, not percentages of objects or data values. Security is excluded.</p>'+
      '<h3>Missing values</h3><p><strong>Not scored</strong> means no saved pair score, often because blocking excluded the pair. <strong>Unavailable</strong> means a score cannot be assessed from the current evidence. Neither means 0%. A complete scan with no roles is <strong>Not applicable</strong> for security; overall then equals schema.</p>'+
      '<h3>Comparison scope</h3><p><strong>Cross-workspace only</strong> includes pairs with known, different workspace IDs. Candidate and group totals follow this scope, while catalog model totals remain unchanged. Groups are rebuilt from qualifying links in scope, and their highest pair score uses those same links. Same-workspace members may still connect through another workspace.</p>'+
      '<p>Map cells marked <strong>Outside scope</strong> are excluded comparisons, not missing or zero scores. Models with unknown workspace IDs are excluded from cross-workspace comparisons. Manual Compare can inspect any explicitly selected pair without changing the scope.</p>'+
      '<h3>Review limits</h3><p>Possible duplicates are candidates, not verified replacements. Scores do not establish equal data, calculation results, effective permissions or retirement safety. Groups are connected by qualifying pairs; not every pair in a group qualifies.</p>'+
      '<p>Linked reports are observed direct dependencies in the scanned scope, not usage. Partial scans show known links only. Zero reports does not prove a model is unused.</p>'+
      ((REPORTS.scans||[]).length?'<h3>Report workspace scans</h3><div>'+REPORTS.scans.map(function(scan){return '<div class="scan-row"><span>'+esc(scan.workspace)+'</span><span>'+esc(scan.status)+(scan.errorType?' / '+esc(scan.errorType):'')+'</span><span>'+esc(scan.reportCount)+' reports</span></div>';}).join('')+'</div>':'')+'</dialog>';
  }
  function workspaceKey(id){var entry=model(id);return modelWorkspaceId(id)||String(entry.workspace||'Unknown');}
  function workspaceOptions(){
    var entries=new Map();MLIST.forEach(function(entry){var key=workspaceKey(entry.id);if(!entries.has(key)){entries.set(key,entry.workspace||'Unknown');}});
    var options=Array.from(entries.entries()),counts=new Map();
    options.forEach(function(entry){counts.set(entry[1],(counts.get(entry[1])||0)+1);});
    return options.map(function(entry){return [entry[0],entry[1]+(counts.get(entry[1])>1?' ['+entry[0]+']':'')];}).sort(function(first,second){return first[1].localeCompare(second[1])||first[0].localeCompare(second[0]);});
  }
  function reviewSelection(){
    var all=candidates(),query=state.search.trim().toLowerCase();
    var base=all.filter(function(item){
      var pair=item.pair;
      return (!query||[model(pair.idA).name,model(pair.idA).workspace,model(pair.idB).name,model(pair.idB).workspace].join(' ').toLowerCase().includes(query))&&
        (state.workspace==='all'||workspaceKey(pair.idA)===state.workspace||workspaceKey(pair.idB)===state.workspace)&&
        (state.security==='all'||pair.securityStatus===state.security)&&
        (state.powerQuery==='all'||(state.powerQuery==='differs'?powerQueryDiffers(pair):state.powerQuery==='matches'?pair.powerQueryStatus==='compared'&&!powerQueryDiffers(pair):pair.powerQueryStatus===state.powerQuery));
    });
    var counts={duplicate:0,containment:0,overlap:0};base.forEach(function(item){counts[item.category.kind]++;});
    var rows=base.filter(function(item){return state.filter==='all'||item.category.kind===state.filter;});
    rows.sort(function(first,second){
      var order=0;
      if(state.sort==='rank'){order=first.category.rank-second.category.rank||second.category.primary-first.category.primary;}
      else{
        var left=first.pair[state.sort],right=second.pair[state.sort];
        if(!validScore(left)||!validScore(right)){order=validScore(left)?-1:(validScore(right)?1:0);}
        else{order=(left-right)*(state.sortDirection==='asc'?1:-1);}
      }
      return order||pairKey(first.pair.idA,first.pair.idB).localeCompare(pairKey(second.pair.idA,second.pair.idB));
    });
    return {all:all,base:base,rows:rows,counts:counts};
  }
  function scoreCellHTML(value,label){
    if(!validScore(value)){return '<span class="cell-value unavailable">'+esc(label||'Unavailable')+'</span>';}
    return '<span class="cell-value">'+score(value)+'</span><div class="cell-bar" aria-hidden="true"><span style="width:'+(value*100)+'%"></span></div>';
  }

  function metricHTML(label,value,unavailableLabel){
    var available=validScore(value),percent=available?Math.round(value*1000)/10:null;
    var meter=available?'<div class="metric-meter" role="meter" aria-label="'+esc(label)+'" aria-valuemin="0" aria-valuemax="100" aria-valuenow="'+percent+'" aria-valuetext="'+score(value)+'"><span class="metric-fill" style="width:'+percent+'%"></span></div>':'<div class="metric-meter unavailable" aria-hidden="true"></div>';
    return '<div class="evidence-item"><dt>'+esc(label)+'</dt><dd><span class="metric-value'+(available?'':' no-value')+'">'+(available?score(value):esc(unavailableLabel||'Unavailable'))+'</span>'+meter+'<span class="metric-scale" aria-hidden="true"><span>0%</span><span>100%</span></span></dd></div>';
  }
  function schemaSignalsHTML(pair){
    var overlap=pair.jaccard||{},hasMeasures=(model(pair.idA).measures||[]).length>0&&(model(pair.idB).measures||[]).length>0;
    var items=[['Table-name overlap',overlap.tables],['Column-name overlap',overlap.columns],['Measure-name overlap',overlap.measures],[hasMeasures?'DAX text similarity':'Model text similarity',pair.daxCosine],['Relationship-link overlap',overlap.relationships],['Source-definition overlap',overlap.datasources],['Power Query similarity',pair.powerQuery,pair.powerQueryStatus==='not_applicable'?'Not applicable':'Unavailable']];
    return '<dl class="evidence-grid">'+items.map(function(item){return metricHTML(item[0],item[1],item[2]);}).join('')+'</dl>';
  }
  function evidenceHTML(pair){
    if(!pair){return '<p class="plain">No saved pair score.</p>';}
    var securityComponents=(pair.securityEvidence||{}).components||{},securityWeights=(pair.securityEvidence||{}).effective_weights||{};
    var securityMetrics=pair.securityStatus==='match'||pair.securityStatus==='different'?'<dl class="evidence-grid">'+metricHTML('Role definitions',securityComponents.role_definitions)+metricHTML('RLS propagation',securityComponents.rls_propagation,'Not applicable')+'</dl><p class="plain">Effective security weights: '+Object.keys(securityWeights).map(function(key){return esc(key==='role_definitions'?'Role definitions':'RLS propagation')+' '+score(securityWeights[key]);}).join(' · ')+'.</p>':'';
    return '<section class="score-section"><h4>Overall '+score(pair.combined)+'</h4><p class="plain">'+esc(blendText(pair))+'</p></section>'+
      '<section class="score-section"><h4>Schema signals</h4>'+schemaSignalsHTML(pair)+'</section>'+
      '<section class="score-section"><h4>Security signals</h4>'+securityMetrics+'</section><button class="status-link" data-help>Calculation definitions</button>';
  }
  function pairIndicatorsHTML(pair){
    return '<div class="pair-indicators" role="region" aria-label="Schema similarity indicators for '+esc(pairModelLabel(pair.idA,pair.idB))+' and '+esc(pairModelLabel(pair.idB,pair.idA))+'">'+schemaSignalsHTML(pair)+'</div>';
  }
  function candidateHTML(item,index){
    var pair=item.pair,key=pairKey(pair.idA,pair.idB),expanded=!!state.expandedPairs[key],detailsId='pair-indicators-'+encodeURIComponent(key);
    function identityCell(id,disclosure){var entry=model(id),name=esc(entry.name);if(disclosure){name='<button class="pair-disclosure" data-expand-pair="'+esc(key)+'" aria-expanded="'+expanded+'" aria-controls="'+esc(detailsId)+'" aria-label="Score indicators for '+esc(pairModelLabel(pair.idA,pair.idB))+' and '+esc(pairModelLabel(pair.idB,pair.idA))+'" title="'+(expanded?'Hide':'Show')+' score indicators"><span>'+name+'</span><span class="row-chevron" aria-hidden="true">&#8250;</span></button>';}return '<td><div class="model-name">'+name+'</div><div class="workspace">'+esc(entry.workspace||'Unknown workspace')+'</div></td>';}
    return '<tr data-pair-key="'+esc(key)+'">'+identityCell(pair.idA,true)+identityCell(pair.idB,false)+
      '<td data-score="combined">'+scoreCellHTML(pair.combined)+'</td><td data-score="schema">'+scoreCellHTML(pair.schema)+'</td><td data-score="security">'+scoreCellHTML(pair.security,pair.securityStatus==='not_applicable'?'Not applicable':'Unavailable')+'</td>'+
      '<td><div class="finding-label">'+esc(item.category.label)+'</div>'+securityTagHTML(pair)+'</td>'+
      '<td><button class="button icon-button" data-compare-a="'+esc(pair.idA)+'" data-compare-b="'+esc(pair.idB)+'" data-open-section="'+relevantSection(pair)+'" title="Compare models" aria-label="Compare '+esc(pairModelLabel(pair.idA,pair.idB))+' and '+esc(pairModelLabel(pair.idB,pair.idA))+'"><span aria-hidden="true">&#8594;</span></button></td></tr>'+
      '<tr class="pair-details" id="'+esc(detailsId)+'"'+(expanded?'':' hidden')+'><td colspan="7">'+(expanded?pairIndicatorsHTML(pair):'')+'</td></tr>';
  }
  function reviewQueueHTML(){
    var selection=reviewSelection(),total=selection.rows.length,pages=Math.max(1,Math.ceil(total/state.pageSize));
    state.page=Math.max(1,Math.min(state.page,pages));
    var start=(state.page-1)*state.pageSize,rows=selection.rows.slice(start,start+state.pageSize);
    var findingOptions=[['all','All findings',selection.base.length],['duplicate',LABELS.duplicate,selection.counts.duplicate],['containment',LABELS.containment,selection.counts.containment],['overlap',LABELS.overlap,selection.counts.overlap]];
    var options=root.querySelector('[data-review-filter="filter"]');
    if(options){options.innerHTML=findingOptions.map(function(entry){return '<option value="'+entry[0]+'"'+(state.filter===entry[0]?' selected':'')+'>'+entry[1]+' ('+entry[2]+')</option>';}).join('');}
    function heading(key,label){var active=state.sort===key,sort=active?(key==='rank'?'other':state.sortDirection==='asc'?'ascending':'descending'):'none';return '<th scope="col" aria-sort="'+sort+'"><button class="table-sort'+(active?' active':'')+'" data-sort="'+key+'">'+label+'<span aria-hidden="true">'+(active&&key!=='rank'?(state.sortDirection==='asc'?'&#8593;':'&#8595;'):'&#8597;')+'</span></button></th>';}
    var table='<div class="table-scroll" tabindex="0" role="region" aria-label="Candidate pair scores"><table class="pair-table"><colgroup><col class="identity-col"><col class="identity-col"><col class="score-col"><col class="score-col"><col class="score-col"><col class="finding-col"><col class="action-col"></colgroup><thead><tr><th scope="col">Model A</th><th scope="col">Model B</th>'+heading('combined','Overall')+heading('schema','Schema')+heading('security','Security')+heading('rank','Finding')+'<th scope="col"><span aria-label="Actions"></span></th></tr></thead><tbody>'+rows.map(candidateHTML).join('')+'</tbody></table></div>';
    var empty=!MLIST.length?'No cataloged models. Run notebook 001.':MLIST.length<2?'Two models are required.':!selection.all.length?(state.crossOnly?'No cross-workspace candidates meet the current thresholds.':'No candidates meet the current thresholds.'):'No candidates match these filters.';
    return '<div class="table-summary"><h2>Candidate pairs</h2><div class="control-row"><span role="status">'+total+' of '+selection.all.length+' candidates</span><button class="status-link" data-clear-filters>Clear filters</button></div></div>'+(rows.length?table:'<div class="empty">'+empty+'</div>')+
      '<div class="pager"><span>Showing '+(total?start+1:0)+'-'+Math.min(start+state.pageSize,total)+' of '+total+'</span><div class="control-row"><label>Rows<select data-page-size aria-label="Rows per page">'+[25,50,100].map(function(size){return '<option'+(size===state.pageSize?' selected':'')+'>'+size+'</option>';}).join('')+'</select></label><button class="button icon-button" data-page="previous" aria-label="Previous page" title="Previous page"'+(state.page===1?' disabled':'')+'>&#8249;</button><span>'+state.page+' / '+pages+'</span><button class="button icon-button" data-page="next" aria-label="Next page" title="Next page"'+(state.page===pages?' disabled':'')+'>&#8250;</button></div></div>';
  }
  function reviewHTML(){
    var all=candidates(),groups=buildGroups();
    return '<div class="stats" aria-label="Catalog model count and comparison-scope totals"><div class="stat"><strong>'+MLIST.length+'</strong><span>Catalog models</span></div><div class="stat"><strong>'+all.length+'</strong><span>'+(state.crossOnly?'Cross-workspace pairs':'Candidate pairs')+'</span></div><div class="stat"><strong>'+groups.length+'</strong><span>Duplicate groups</span></div></div>'+statusStripHTML()+
      '<div class="review-filters"><label class="search-label">Search<input class="search" type="text" aria-label="Search review candidates" placeholder="Models or workspaces" value="'+esc(state.search)+'"></label>'+
      '<label>Finding<select data-review-filter="filter" aria-label="Finding"><option value="all">All findings</option></select></label>'+
      '<label>Workspace<select data-review-filter="workspace" aria-label="Workspace"><option value="all">All workspaces</option>'+workspaceOptions().map(function(entry){return '<option value="'+esc(entry[0])+'"'+(entry[0]===state.workspace?' selected':'')+'>'+esc(entry[1])+'</option>';}).join('')+'</select></label>'+
      '<label>Security<select data-review-filter="security" aria-label="Security state">'+[['all','All states'],['different','Differs'],['match','Matches'],['not_applicable','No model roles'],['unknown','Not assessed']].map(function(entry){return '<option value="'+entry[0]+'"'+(entry[0]===state.security?' selected':'')+'>'+entry[1]+'</option>';}).join('')+'</select></label>'+
      '<label>Power Query<select data-review-filter="powerQuery" aria-label="Power Query state">'+[['all','All states'],['differs','Differs'],['matches','Matches'],['not_applicable','No M on either']].map(function(entry){return '<option value="'+entry[0]+'"'+(entry[0]===state.powerQuery?' selected':'')+'>'+entry[1]+'</option>';}).join('')+'</select></label></div><div class="queue"></div>';
  }

  function selectionLabel(id){var entry=model(id),duplicate=MLIST.some(function(other){return other.id!==id&&other.name===entry.name&&other.workspace===entry.workspace;});return entry.name+' / '+(entry.workspace||'Unknown workspace')+(duplicate?' ['+id+']':'');}
  function groupOptions(members,selected){return members.map(function(id){return '<option value="'+esc(id)+'"'+(id===selected?' selected':'')+'>'+esc(selectionLabel(id))+'</option>';}).join('');}
  function modelSecurityText(id){var security=model(id).security||{};return security.status==='complete'?(security.roleCount?security.roleCount+' security role'+(security.roleCount===1?'':'s'):'No model-level security roles'):'Security not assessed';}
  function groupKey(group){return JSON.stringify(group.members.slice().sort());}
  function groupSelection(group){
    var key=groupKey(group),chosen=state.groupSelections[key];
    if(!chosen||!group.members.includes(chosen.a)){chosen={a:group.strongest.idA,b:group.strongest.idB};}
    var options=group.members.filter(function(id){return id!==chosen.a&&inComparisonScope(chosen.a,id);});
    if(!options.includes(chosen.b)){chosen={a:chosen.a,b:options[0]||''};}
    state.groupSelections[key]=chosen;return {chosen:chosen,options:options};
  }
  function shortReportCount(id){return reportsAvailable()?reportsFor(id).length+(REPORTS.status==='partial'?' known':' reports'):'Unknown';}
  function groupSecurityHTML(group){
    var labels=[];
    if(group.securityDifferent){labels.push('<span class="security-tag warning">Security differs</span>');}
    if(group.securityUnknown){labels.push('<span class="security-tag warning">Not assessed</span>');}
    return '<span class="security-status-pair">'+(labels.length?labels.join(''):'<span class="security-tag">Matching definitions</span>')+'</span>';
  }
  function groupResultsHTML(){
    var all=buildGroups(),query=state.groupSearch.trim().toLowerCase();
    var groups=all.map(function(group,index){return {group:group,label:'Group '+(index+1),key:groupKey(group)};}).filter(function(entry){return !query||entry.group.members.some(function(id){var entryModel=model(id);return (entryModel.name+' '+entryModel.workspace).toLowerCase().includes(query);});});
    if(!groups.length){return '<div class="empty">'+(all.length?'No groups match this search.':state.crossOnly?'No cross-workspace duplicate groups at this cutoff.':'No possible-duplicate groups.')+'</div>';}
    if(!groups.some(function(entry){return entry.key===state.selectedGroup;})){state.selectedGroup=groups[0].key;}
    var selected=groups.find(function(entry){return entry.key===state.selectedGroup;});
    var table='<div class="table-scroll" tabindex="0" role="region" aria-label="Possible duplicate groups"><table class="group-table"><thead><tr><th scope="col">Group</th><th scope="col">Models</th><th scope="col">Workspaces</th><th scope="col">Highest pair score</th><th scope="col">Security</th></tr></thead><tbody>'+
      groups.map(function(entry){var group=entry.group,active=entry.key===state.selectedGroup;return '<tr class="'+(active?'selected':'')+'"><td><button class="status-link" data-select-group="'+esc(entry.key)+'" aria-expanded="'+active+'" aria-controls="group-detail">'+entry.label+'</button></td><td>'+group.members.length+'</td><td>'+new Set(group.members.map(workspaceKey)).size+'</td><td>'+scoreCellHTML(group.strongest.combined)+'</td><td>'+groupSecurityHTML(group)+'</td></tr>';}).join('')+'</tbody></table></div>';
    var selection=groupSelection(selected.group),chosen=selection.chosen;
    var members=selected.group.members.map(function(id){var entry=model(id),security=modelSecurityText(id).replace('No model-level security roles','No model roles').replace('Security not assessed','Not assessed');return '<li><span class="model-name">'+esc(entry.name)+'</span><span class="workspace">'+esc(entry.workspace)+'</span><span class="member-security">'+esc(security)+'</span><span class="member-reports">'+esc(shortReportCount(id))+'</span></li>';}).join('');
    return '<div class="table-summary"><span role="status">'+groups.length+' of '+all.length+' groups</span><button class="status-link" data-help>Group definitions</button></div>'+table+
      '<section id="group-detail" class="group-detail" aria-labelledby="group-detail-title"><div class="group-head"><h3 id="group-detail-title">'+selected.label+' / '+selected.group.members.length+' models</h3>'+groupSecurityHTML(selected.group)+'</div><ul class="group-members">'+members+'</ul>'+
      '<div class="group-pickers"><label>Model A<select data-group="'+esc(selected.key)+'" data-group-side="a">'+groupOptions(selected.group.members,chosen.a)+'</select></label><label>Model B<select data-group="'+esc(selected.key)+'" data-group-side="b">'+groupOptions(selection.options,chosen.b)+'</select></label><button class="button primary" data-group-compare="'+esc(selected.key)+'"'+(!chosen.b?' disabled':'')+'>Compare</button></div></section>';
  }
  function groupsHTML(){
    return '<h2>Possible duplicate groups</h2>'+statusStripHTML()+'<div class="explorer-toolbar"><label>Search<input data-group-search type="text" aria-label="Search groups" placeholder="Models or workspaces" value="'+esc(state.groupSearch)+'"></label></div><div id="group-results">'+groupResultsHTML()+'</div>';
  }

  function indexBy(list){var out={};(list||[]).forEach(function(x){out[x.key]=x;});return out;}
  function unionKeys(a,b){var out={};Object.keys(a).forEach(function(k){out[k]=1;});Object.keys(b).forEach(function(k){out[k]=1;});return Object.keys(out).sort();}
  function statusOf(key,a,b){return a[key]&&b[key]?'shared':(a[key]?'onlyA':'onlyB');}
  function statusLabel(status,firstName,secondName){return status==='onlyA'?'Only in '+firstName:(status==='onlyB'?'Only in '+secondName:(status==='changed'?'Different formula text':'Matching entry'));}
  function sectionHTML(id,title,summary,rows){var open=!!state.openSections[id];return '<section class="diff-section"><button class="section-button" data-section="'+id+'" aria-expanded="'+open+'" aria-controls="section-'+id+'"><span>'+title+'</span><span class="muted">'+summary+'</span></button><div class="section-body" id="section-'+id+'"'+(open?'':' hidden')+'>'+((rows&&rows.length)?rows.join(''):'<div class="muted">No cataloged entries to show.</div>')+'</div></section>';}
  function rowHTML(st,text,aName,bName,detail){return '<div class="diff-row '+st+'"><span class="status">'+esc(statusLabel(st,aName,bName))+'</span><span>'+text+'</span></div>'+(detail?'<details class="formula-detail"><summary>Formula text</summary>'+detail+'</details>':'');}
  function compareData(a,b,firstName,secondName){
    firstName=firstName||a.name;secondName=secondName||b.name;
    var result={sections:{},shared:0,differences:0};
    var ta=indexBy(a.tables),tb=indexBy(b.tables),ca=indexBy(a.columns),cb=indexBy(b.columns),ma=indexBy(a.measures),mb=indexBy(b.measures),ra=indexBy(a.relationships),rb=indexBy(b.relationships),da=indexBy(a.datasources),db=indexBy(b.datasources),qa=indexBy(a.queries),qb=indexBy(b.queries);
    function simple(id,left,right,format){var rows=[],shared=0,diff=0;unionKeys(left,right).forEach(function(k){var st=statusOf(k,left,right);if(st==='shared'){shared++;}else{diff++;}if(!state.cmpDiffOnly||st!=='shared'){rows.push(rowHTML(st,format(left[k]||right[k]),firstName,secondName));}});result.sections[id]={rows:rows,shared:shared,diff:diff};result.shared+=shared;result.differences+=diff;}
    simple('tables',ta,tb,function(x){return esc(x.name);});
    simple('columns',ca,cb,function(x){return esc(x.table)+'['+esc(x.name)+']';});
    var measureRows=[],measureShared=0,measureDiff=0;unionKeys(ma,mb).forEach(function(k){var ia=ma[k],ib=mb[k],st;if(ia&&ib){st=ia.daxHash===ib.daxHash?'shared':'changed';}else{st=ia?'onlyA':'onlyB';}if(st==='shared'){measureShared++;}else{measureDiff++;}if(!state.cmpDiffOnly||st!=='shared'){var item=ia||ib,detail='';if(st==='changed'){detail='<div class="dax"><strong>'+esc(firstName)+'</strong>\n'+esc(daxText(ia))+'\n\n<strong>'+esc(secondName)+'</strong>\n'+esc(daxText(ib))+'</div>';}else if(st!=='shared'){detail='<div class="dax">'+esc(daxText(item))+'</div>';}measureRows.push(rowHTML(st,esc(item.name),firstName,secondName,detail));}});result.sections.measures={rows:measureRows,shared:measureShared,diff:measureDiff};result.shared+=measureShared;result.differences+=measureDiff;
    simple('relationships',ra,rb,function(x){return esc(x.from)+' → '+esc(x.to);});
    simple('datasources',da,db,function(x){return esc(x.name);});
    var queryRows=[],queryShared=0,queryDiff=0;unionKeys(qa,qb).forEach(function(k){var ia=qa[k],ib=qb[k],st;if(ia&&ib){st=ia.mHash===ib.mHash?'shared':'changed';}else{st=ia?'onlyA':'onlyB';}if(st==='shared'){queryShared++;}else{queryDiff++;}if(!state.cmpDiffOnly||st!=='shared'){var item=ia||ib,detail='';if(st==='changed'){detail='<div class="dax"><strong>'+esc(firstName)+'</strong>\n'+esc(mText(ia))+'\n\n<strong>'+esc(secondName)+'</strong>\n'+esc(mText(ib))+'</div>';}else if(st!=='shared'){detail='<div class="dax">'+esc(mText(item))+'</div>';}queryRows.push(rowHTML(st,esc(item.name),firstName,secondName,detail).replace('<summary>Formula text</summary>','<summary>M query</summary>'));}});result.sections.queries={rows:queryRows,shared:queryShared,diff:queryDiff};result.shared+=queryShared;result.differences+=queryDiff;
    return result;
  }
  function securityFacts(role){
    var facts=Object.create(null);
    if(!role){return facts;}
    facts.permission={label:'Model permission',value:role.model_permission};
    [['rls','RLS filter'],['table_ols','Table OLS'],['column_ols','Column OLS (CLS)']].forEach(function(family){
      (role[family[0]]||[]).forEach(function(fact){var target=fact.slice(0,-1),key=JSON.stringify([family[0],target]);facts[key]={label:family[1]+': '+target.join(' / '),value:fact[fact.length-1]};});
    });
    return facts;
  }
  function securityRows(left,right,firstName,secondName){
    var rows=[],shared=0,different=0;
    unionKeys(left,right).forEach(function(key){
      var first=left[key],second=right[key],status=first&&second?(first.value===second.value?'shared':'changed'):(first?'onlyA':'onlyB');
      if(status==='shared'){shared++;}else{different++;}
      if(state.cmpDiffOnly&&status==='shared'){return;}
      var label=status==='changed'?'Different rule':(status==='shared'?'Matching rule':('Only in '+(first?firstName:secondName)));
      var detail=status==='shared'?esc(first.value):((first?'<strong>'+esc(firstName)+'</strong>\n'+esc(first.value):'')+(first&&second?'\n\n':'')+(second?'<strong>'+esc(secondName)+'</strong>\n'+esc(second.value):''));
      rows.push('<details class="rule-row"><summary><span class="status">'+esc(label)+'</span><span class="security-detail">'+esc((first||second).label)+'</span></summary><div class="dax">'+detail+'</div></details>');
    });
    return {rows:rows,shared:shared,different:different};
  }
  function propagationFacts(security){
    var facts=Object.create(null);
    (security.propagation||[]).forEach(function(fact){facts[JSON.stringify(fact)]={label:'RLS propagation: '+fact[0]+'['+fact[1]+'] to '+fact[2]+'['+fact[3]+']',value:'Active: '+(fact[4]?'Yes':'No')+'; cross-filter: '+fact[5]+'; security filter: '+fact[6]+'; cardinality: '+fact[7]+' to '+fact[8]};});
    return facts;
  }
  function securityInventoryHTML(id){
    var security=model(id).security||{},name=pairModelLabel(id,id===state.cmpA?state.cmpB:state.cmpA);
    if(security.status!=='complete'){return '<div class="security-role"><h4>'+esc(name)+'</h4><span class="security-tag warning">Not assessed</span></div>';}
    if(!security.roles.length){return '<div class="security-role"><h4>'+esc(name)+'</h4><span class="security-tag">No model roles</span></div>';}
    return security.roles.map(function(role){var facts=securityFacts(role);return '<div class="security-role"><h4>'+esc(name)+' / '+esc(role.name)+'</h4>'+Object.keys(facts).map(function(key){return '<div class="plain">'+esc(facts[key].label)+'</div><div class="dax">'+esc(facts[key].value)+'</div>';}).join('')+'</div>';}).join('');
  }
  function securityCompareHTML(pair){
    var first=model(state.cmpA).security||{},second=model(state.cmpB).security||{};
    if(!pair||pair.securityStatus==='unknown'||first.status!=='complete'||second.status!=='complete'){
      return sectionHTML('security','Security definitions',securityLabel(pair),['<span class="security-tag warning">No aligned comparison</span>'+securityInventoryHTML(state.cmpA)+securityInventoryHTML(state.cmpB)]);
    }
    if(pair.securityStatus==='not_applicable'){return sectionHTML('security','Security definitions','Not applicable',['<span class="security-tag">No model roles in either complete scan</span>']);}
    var reversed=pair.idA!==state.cmpA,evidence=pair.securityEvidence||{},blocks=[],shared=0,different=0;
    var firstName=pairModelLabel(state.cmpA,state.cmpB),secondName=pairModelLabel(state.cmpB,state.cmpA);
    function addRole(roleA,roleB,roleScore){
      var compared=securityRows(securityFacts(roleA),securityFacts(roleB),firstName,secondName);shared+=compared.shared;different+=compared.different;
      if(state.cmpDiffOnly&&!compared.different){return;}
      var title=roleA&&roleB?'Roles: '+roleA.name+' / '+roleB.name:('Unmatched role: '+(roleA||roleB).name+' ('+(roleA?firstName:secondName)+')');
      blocks.push('<div class="security-role"><h4>'+esc(title)+'</h4><div class="plain">Role similarity: '+score(roleScore)+'</div>'+compared.rows.join('')+'</div>');
    }
    (evidence.role_matches||[]).forEach(function(match){addRole(first.roles[reversed?match.role_b:match.role_a],second.roles[reversed?match.role_a:match.role_b],match.score);});
    (evidence[reversed?'unmatched_roles_b':'unmatched_roles_a']||[]).forEach(function(index){addRole(first.roles[index],null,0);});
    (evidence[reversed?'unmatched_roles_a':'unmatched_roles_b']||[]).forEach(function(index){addRole(null,second.roles[index],0);});
    var propagation=securityRows(propagationFacts(first),propagationFacts(second),firstName,secondName);shared+=propagation.shared;different+=propagation.different;
    if(propagation.rows.length){blocks.push('<div class="security-role"><h4>RLS propagation</h4>'+propagation.rows.join('')+'</div>');}
    if(!blocks.length){blocks.push('<span class="security-tag">No differing security rules</span>');}
    return sectionHTML('security','Security definitions',shared+' matching rules · '+different+' different',blocks);
  }
  function relationshipSummary(pair){if(!pair){return 'Not scored';}var category=classify(pair);return category?category.label:'Below review cutoffs or unavailable';}
  function containmentSummary(pair){
    if(!pair){return 'Not scored';}
    var context=coverageContext(pair);
    if(!context.strongest){return 'Directional coverage unavailable';}
    if(context.directions.some(function(direction){return !validScore(direction.value);})){return 'Coverage comparison is incomplete: one directional score is unavailable.';}
    if(context.both){return 'Both models meet the schema coverage threshold ('+score(state.thresholds.containment)+').';}
    if(context.qualifies){return 'One direction meets the schema coverage threshold ('+score(state.thresholds.containment)+').';}
    return 'Neither direction meets the schema coverage threshold ('+score(state.thresholds.containment)+').';
  }
  function selectOptions(selected){return MLIST.map(function(entry){return '<option value="'+esc(entry.id)+'"'+(entry.id===selected?' selected':'')+'>'+esc(selectionLabel(entry.id))+'</option>';}).join('');}
  function compareCoverageHTML(pair){
    if(!pair){return '<span class="muted">Coverage: Not scored</span>';}
    var reversed=pair.idA!==state.cmpA;
    var directions=[{source:state.cmpA,target:state.cmpB,label:'A within B',value:reversed?pair.bInA:pair.aInB},
                    {source:state.cmpB,target:state.cmpA,label:'B within A',value:reversed?pair.aInB:pair.bInA}];
    return '<dl class="coverage-list" aria-label="Directional schema coverage">'+directions.map(function(direction){var full=pairModelLabel(direction.source,direction.target)+' within '+pairModelLabel(direction.target,direction.source);return '<div><dt title="'+esc(full)+'">Schema coverage: '+direction.label+'</dt><dd aria-label="'+esc(full)+': '+score(direction.value)+'">'+score(direction.value)+'</dd></div>';}).join('')+'</dl>';
  }
  function compareHTML(){
    var title='<div class="compare-title">'+(state.returnView?'<button class="button icon-button" data-back title="Back to '+esc(state.returnView.tab)+'" aria-label="Back to '+esc(state.returnView.tab)+'"><span aria-hidden="true">&#8592;</span></button>':'')+'<h2>Compare models</h2></div>';
    if(MLIST.length<2){return title+'<div class="empty">Two cataloged models are required.</div>';}
    var first=model(state.cmpA),second=model(state.cmpB),pair=pairMap()[pairKey(state.cmpA,state.cmpB)];
    var controls='<div class="compare-controls"><label>Model A<select data-compare-select="a" title="'+esc(pairModelLabel(state.cmpA,state.cmpB))+'">'+selectOptions(state.cmpA)+'</select></label><button class="button icon-button" data-swap title="Swap models" aria-label="Swap selected models"><span aria-hidden="true">&#8644;</span></button><label>Model B<select data-compare-select="b" title="'+esc(pairModelLabel(state.cmpB,state.cmpA))+'">'+selectOptions(state.cmpB)+'</select></label></div>';
    if(state.cmpA===state.cmpB){return title+controls+'<div class="empty">Choose two different models.</div>';}
    var diff=compareData(first,second,pairModelLabel(state.cmpA,state.cmpB),pairModelLabel(state.cmpB,state.cmpA));
    function sec(id,title){var entry=diff.sections[id]||{rows:[],shared:0,diff:0};return sectionHTML(id,title,entry.shared+' matching / '+entry.diff+' different',entry.rows.length?entry.rows:['<div class="muted">'+(state.cmpDiffOnly?'No differences.':'No cataloged entries.')+'</div>']);}
    return title+controls+'<div class="compare-summary"><div class="compare-flags"><span class="finding-label">'+esc(relationshipSummary(pair))+'</span>'+securityTagHTML(pair)+(!inComparisonScope(state.cmpA,state.cmpB)?'<span class="scope-note">Outside scope: '+scopeExclusion(state.cmpA,state.cmpB)+'</span>':'')+'<label><input type="checkbox" data-diff-only'+(state.cmpDiffOnly?' checked':'')+'> Differences only</label></div>'+scoreSummaryHTML(pair)+
      '<div class="comparison-context">'+compareCoverageHTML(pair)+'<div class="schema-counts">Schema entries<br><strong>'+diff.shared+' matching / '+diff.differences+' different</strong></div></div></div>'+
      securityCompareHTML(pair)+sec('tables','Tables')+sec('columns','Columns')+sec('measures','Measures')+sec('queries','Power Query')+sec('relationships','Relationships')+sec('datasources','Sources')+reportCompareHTML()+sectionHTML('scores','Score breakdown','Schema / security signals',[evidenceHTML(pair)]);
  }

  function mapSelection(){
    var query=state.mapSearch.trim().toLowerCase();
    var models=MLIST.filter(function(entry){return (!query||(entry.name+' '+entry.workspace).toLowerCase().includes(query))&&(state.mapWorkspace==='all'||workspaceKey(entry.id)===state.mapWorkspace);});
    var pages=Math.max(1,Math.ceil(models.length/state.mapSize));state.mapPage=Math.max(1,Math.min(state.mapPage,pages));
    var start=(state.mapPage-1)*state.mapSize;return {models:models,visible:models.slice(start,start+state.mapSize),pages:pages,start:start};
  }
  function mapResultsHTML(){
    var selection=mapSelection(),visible=selection.visible;
    if(!visible.length){return '<div class="empty">'+(MLIST.length?'No models match these filters.':'No cataloged models.')+'</div>';}
    var pmap=pairMap(),head='<thead><tr><th scope="col">Model</th>'+visible.map(function(entry){var label=entry.name+' / '+entry.workspace;return '<th class="col" scope="col" title="'+esc(label)+'"><span class="axis-label">'+esc(label)+'</span></th>';}).join('')+'</tr></thead>';
    var rows=visible.map(function(rowModel,rowIndex){var cells=visible.map(function(columnModel,columnIndex){
      var rowLabel=rowModel.name+' (Workspace: '+rowModel.workspace+')',columnLabel=columnModel.name+' (Workspace: '+columnModel.workspace+')',pairLabel=rowLabel+' and '+columnLabel;
      if(rowIndex===columnIndex){return '<td><span class="matrix-cell diagonal" title="'+esc(rowLabel)+': same model" aria-label="'+esc(rowLabel)+': same model">&#8212;</span></td>';}
      if(!inComparisonScope(rowModel.id,columnModel.id)){var excluded=pairLabel+': Outside scope ('+scopeExclusion(rowModel.id,columnModel.id)+')';return '<td><button class="matrix-cell outside" aria-disabled="true" data-map-excluded data-map-detail="'+esc(excluded)+'" title="'+esc(excluded)+'" aria-label="'+esc(excluded)+'">&#215;</button></td>';}
      var pair=pmap[pairKey(rowModel.id,columnModel.id)];
      if(!pair){return '<td><button class="matrix-cell unscored" data-map-a="'+esc(rowModel.id)+'" data-map-b="'+esc(columnModel.id)+'" title="'+esc(pairLabel)+': Not scored" aria-label="Compare '+esc(pairLabel)+', Not scored" data-map-detail="'+esc(pairLabel)+': Not scored"><span aria-hidden="true">&#183;</span></button></td>';}
      var tier=tierAt(pair.combined),available=validScore(pair.combined),style=!available?'unavailable':(tier==='duplicate'?'duplicate':(tier==='similar'?'high':'low'));
      var description='Overall '+score(pair.combined)+'; schema '+score(pair.schema)+'; security '+securityScoreText(pair)+'; '+securityLabel(pair);
      return '<td><button class="matrix-cell '+style+'" data-map-a="'+esc(rowModel.id)+'" data-map-b="'+esc(columnModel.id)+'" data-map-detail="'+esc(pairLabel)+': '+esc(description)+'" title="'+esc(pairLabel)+': '+esc(description)+'" aria-label="Compare '+esc(pairLabel)+', '+esc(description)+'">'+(available?Math.round(pair.combined*100)+'%':'?')+'</button></td>';
    }).join('');return '<tr><th scope="row" title="'+esc(rowModel.name)+' / '+esc(rowModel.workspace)+'">'+esc(rowModel.name)+'<div class="workspace">'+esc(rowModel.workspace)+'</div></th>'+cells+'</tr>';}).join('');
    return '<div class="pager"><span role="status">Showing '+(selection.start+1)+'-'+(selection.start+visible.length)+' of '+selection.models.length+' filtered models / '+MLIST.length+' catalog models</span><div class="control-row"><button class="button icon-button" data-map-page="previous" aria-label="Previous model slice" title="Previous model slice"'+(state.mapPage===1?' disabled':'')+'>&#8249;</button><span>'+state.mapPage+' / '+selection.pages+'</span><button class="button icon-button" data-map-page="next" aria-label="Next model slice" title="Next model slice"'+(state.mapPage===selection.pages?' disabled':'')+'>&#8250;</button></div></div>'+
      '<div class="legend" aria-label="Similarity map legend"><span><span class="legend-key duplicate"></span> Possible duplicates</span><span><span class="legend-key high"></span> Shared structure</span><span><span class="legend-key low"></span> Below cutoff (including 0%)</span><span><span class="legend-key unscored"></span> Not scored (&#183;)</span><span><span class="legend-key unavailable"></span> Unavailable (?)</span><span>&#8212; Same model</span>'+(state.crossOnly?'<span>&#215; Outside scope</span>':'')+'</div>'+
      '<div class="matrix-container" tabindex="0" role="region" aria-label="Overall similarity matrix"><table class="matrix" aria-label="Semantic model overall similarity map">'+head+'<tbody>'+rows+'</tbody></table></div><div class="map-detail" aria-live="polite">'+visible.length+' selected models</div>';
  }
  function mapHTML(){
    return '<h2>Overall similarity map</h2>'+statusStripHTML()+'<div class="explorer-toolbar"><label>Search<input type="text" data-map-search aria-label="Search map models" placeholder="Models or workspaces" value="'+esc(state.mapSearch)+'"></label><label>Workspace<select data-map-workspace aria-label="Map workspace"><option value="all">All workspaces</option>'+workspaceOptions().map(function(entry){return '<option value="'+esc(entry[0])+'"'+(entry[0]===state.mapWorkspace?' selected':'')+'>'+esc(entry[1])+'</option>';}).join('')+'</select></label></div><div id="map-results">'+mapResultsHTML()+'</div>';
  }

  function viewHTML(){var content=state.tab==='review'?reviewHTML():(state.tab==='groups'?groupsHTML():(state.tab==='compare'?compareHTML():(state.tab==='reports'?reportsHTML():mapHTML())));return '<div class="view" role="tabpanel" id="panel-'+state.tab+'" aria-labelledby="tab-'+state.tab+'">'+content+'</div>';}
  function render(){root.innerHTML=headerHTML()+settingsHTML()+tabsHTML()+viewHTML()+helpHTML();if(state.tab==='review'){root.querySelector('.queue').innerHTML=reviewQueueHTML();}applyTheme(false);wire();}
  function setTab(tab){state.tab=tab;render();}
  function openCompare(a,b,section){
    if(state.tab!=='compare'){
      var focused=document.activeElement,scroll=root.querySelector('.table-scroll, .matrix-container');
      state.returnView={tab:state.tab,scroll:window.scrollY,tableScroll:scroll?scroll.scrollLeft:0,tableTop:scroll?scroll.scrollTop:0,focus:focused?Object.assign({},focused.dataset):{}};
    }
    state.cmpA=String(a);state.cmpB=String(b);state.cmpDiffOnly=true;state.openSections={};
    if(section){state.openSections[section]=true;}
    setTab('compare');root.querySelector('[data-compare-select="a"]').focus({preventScroll:true});root.scrollIntoView({block:'start'});
  }
  function backToResults(){
    var previous=state.returnView;if(!previous){return;}
    setTab(previous.tab);
    var control=Array.from(root.querySelectorAll('button')).find(function(button){var entries=Object.entries(previous.focus);return entries.length&&entries.every(function(entry){return button.dataset[entry[0]]===entry[1];});});
    if(control){control.focus({preventScroll:true});}
    var table=root.querySelector('.table-scroll, .matrix-container');if(table){table.scrollLeft=previous.tableScroll;table.scrollTop=previous.tableTop;}
    window.scrollTo(0,previous.scroll);
    if(control&&window.parent!==window){control.scrollIntoView({block:'nearest',inline:'nearest'});}
  }
  function applyTheme(save){document.documentElement.setAttribute('data-theme',state.theme);if(save){try{localStorage.setItem('sms-theme',state.theme);}catch(e){}}}
  function applyThresholds(reset){
    if(reset!==true){
      var inputs=Array.from(root.querySelectorAll('[data-draft-number]'));
      var invalid=inputs.find(function(input){return !input.checkValidity()||input.value==='';});
      if(invalid){root.querySelector('#threshold-error').hidden=false;invalid.focus();return;}
      inputs.forEach(function(input){state.draft[input.dataset.draftNumber]=Number(input.value)/100;});
    }
    if(!['duplicate','similar','containment'].every(function(key){return validScore(state.draft[key]);})){return;}
    state.thresholds=Object.assign({},state.draft);state.page=1;
    try{localStorage.setItem('sms-thresholds-v2',JSON.stringify(state.thresholds));}catch(e){}
    render();var apply=root.querySelector('[data-apply]');if(apply&&!root.querySelector('#scoring-settings').hidden){apply.focus({preventScroll:true});}
  }

  function togglePairIndicators(row){
    var key=row.dataset.pairKey,button=row.querySelector('[data-expand-pair]'),details=row.nextElementSibling;
    var expanded=!state.expandedPairs[key];state.expandedPairs[key]=expanded;
    button.setAttribute('aria-expanded',String(expanded));button.title=(expanded?'Hide':'Show')+' score indicators';
    details.querySelector('td').innerHTML=expanded?pairIndicatorsHTML(pairMap()[key]):'';details.hidden=!expanded;
  }
  function wireCandidateActions(scope){
    scope.querySelectorAll('[data-evidence]').forEach(function(btn){btn.addEventListener('click',function(){var target=scope.querySelector('#'+btn.dataset.evidence),expanded=btn.getAttribute('aria-expanded')==='true';btn.setAttribute('aria-expanded',String(!expanded));if(target){target.hidden=expanded;}});});
    scope.querySelectorAll('[data-compare-a]').forEach(function(btn){btn.addEventListener('click',function(){openCompare(btn.dataset.compareA,btn.dataset.compareB,btn.dataset.openSection);});});
    scope.querySelectorAll('[data-expand-pair]').forEach(function(button){button.addEventListener('click',function(){togglePairIndicators(button.closest('[data-pair-key]'));});});
    scope.querySelectorAll('[data-pair-key]').forEach(function(row){row.addEventListener('click',function(event){if(event.target.closest('button,a,input,summary,select')){return;}row.querySelector('[data-expand-pair]').focus({preventScroll:true});togglePairIndicators(row);});});
  }

  function refreshQueue(){var queue=root.querySelector('.queue');if(queue){queue.innerHTML=reviewQueueHTML();wireReviewTable(queue);}}
  function wireReviewTable(scope){
    wireCandidateActions(scope);
    scope.querySelectorAll('[data-sort]').forEach(function(button){button.addEventListener('click',function(){var key=button.dataset.sort;state.sortDirection=state.sort===key&&state.sortDirection==='desc'?'asc':'desc';state.sort=key;state.page=1;refreshQueue();root.querySelector('[data-sort="'+key+'"]').focus({preventScroll:true});});});
    scope.querySelectorAll('[data-page]').forEach(function(button){button.addEventListener('click',function(){var direction=button.dataset.page;state.page+=direction==='next'?1:-1;refreshQueue();var target=root.querySelector('[data-page="'+direction+'"]');if(target&&!target.disabled){target.focus({preventScroll:true});}else{root.querySelector('.table-scroll')?.focus({preventScroll:true});}});});
    var pageSize=scope.querySelector('[data-page-size]');if(pageSize){pageSize.addEventListener('change',function(){state.pageSize=Number(pageSize.value);state.page=1;refreshQueue();root.querySelector('[data-page-size]').focus({preventScroll:true});});}
    var clear=scope.querySelector('[data-clear-filters]');if(clear){clear.addEventListener('click',function(){clearReviewFilters();render();root.querySelector('.search').focus({preventScroll:true});});}
  }

  function refreshGroups(){var content=root.querySelector('#group-results');if(content){content.innerHTML=groupResultsHTML();wireGroupControls(content);}}
  function wireGroupControls(scope){
    scope.querySelectorAll('[data-select-group]').forEach(function(button){button.addEventListener('click',function(){var key=button.dataset.selectGroup;state.selectedGroup=key;refreshGroups();var target=Array.from(root.querySelectorAll('[data-select-group]')).find(function(entry){return entry.dataset.selectGroup===key;});if(target){target.focus({preventScroll:true});}});});
    scope.querySelectorAll('[data-group-side]').forEach(function(select){select.addEventListener('change',function(){var key=select.dataset.group,side=select.dataset.groupSide;state.groupSelections[key]=state.groupSelections[key]||{};state.groupSelections[key][side]=select.value;refreshGroups();root.querySelector('[data-group-side="'+side+'"]').focus({preventScroll:true});});});
    scope.querySelectorAll('[data-group-compare]').forEach(function(button){button.addEventListener('click',function(){var selected=state.groupSelections[button.dataset.groupCompare];if(selected&&selected.b&&selected.a!==selected.b&&inComparisonScope(selected.a,selected.b)){openCompare(selected.a,selected.b,relevantSection(pairMap()[pairKey(selected.a,selected.b)]));}});});
  }
  function refreshMap(){var content=root.querySelector('#map-results');if(content){content.innerHTML=mapResultsHTML();wireMapResults(content);}}
  function wireMapResults(scope){
    scope.querySelectorAll('[data-map-detail]').forEach(function(button){
      function details(){var target=root.querySelector('.map-detail');if(target){target.textContent=button.dataset.mapDetail;}}
      button.addEventListener('focus',details);button.addEventListener('pointerenter',details);
      if(button.hasAttribute('data-map-a')){button.addEventListener('click',function(){if(inComparisonScope(button.dataset.mapA,button.dataset.mapB)){openCompare(button.dataset.mapA,button.dataset.mapB,relevantSection(pairMap()[pairKey(button.dataset.mapA,button.dataset.mapB)]));}});}
    });
    scope.querySelectorAll('[data-map-page]').forEach(function(button){button.addEventListener('click',function(){var direction=button.dataset.mapPage;state.mapPage+=direction==='next'?1:-1;refreshMap();var target=root.querySelector('[data-map-page="'+direction+'"]');if(target&&!target.disabled){target.focus({preventScroll:true});}else{root.querySelector('.matrix-container')?.focus({preventScroll:true});}});});
  }

  function openHelp(dialog,trigger){
    function show(bounds){
      if(!dialog.isConnected||!trigger.isConnected||dialog.open){return;}
      var width=Math.min(620,Math.max(0,bounds.width-24));
      dialog.style.width=width+'px';
      dialog.style.maxHeight=Math.min(560,Math.max(0,bounds.height-24))+'px';
      dialog.style.top=(bounds.top+12)+'px';
      dialog.style.left=(bounds.left+(bounds.width-width)/2)+'px';
      dialog.showModal();dialog.scrollTop=0;
      dialog.querySelector('[data-help-close]').focus({preventScroll:true});
    }
    function fallback(){show({top:0,left:0,width:window.innerWidth,height:window.innerHeight});}
    if(typeof IntersectionObserver!=='function'){fallback();return;}
    var observer=new IntersectionObserver(function(entries){
      observer.disconnect();var bounds=entries[0].intersectionRect;
      if(bounds.width>0&&bounds.height>0){show(bounds);}else{fallback();}
    });
    observer.observe(root);
  }

  function wire(){
    var tabs=Array.from(root.querySelectorAll('[data-tab]'));
    tabs.forEach(function(button,index){
      button.addEventListener('click',function(){setTab(button.dataset.tab);});
      button.addEventListener('keydown',function(event){
        var next=index;
        if(event.key==='ArrowRight'){next=(index+1)%tabs.length;}
        else if(event.key==='ArrowLeft'){next=(index+tabs.length-1)%tabs.length;}
        else if(event.key==='Home'){next=0;}
        else if(event.key==='End'){next=tabs.length-1;}
        else{return;}
        event.preventDefault();setTab(tabs[next].dataset.tab);root.querySelector('[data-tab="'+state.tab+'"]').focus();
      });
    });
    var settings=root.querySelector('[data-settings]');if(settings){settings.addEventListener('click',function(){state.settingsOpen=!state.settingsOpen;render();root.querySelector('[data-settings]').focus();});}
    root.querySelectorAll('[data-theme-set]').forEach(function(btn){btn.addEventListener('click',function(){state.theme=btn.dataset.themeSet;applyTheme(true);render();});});
    root.querySelectorAll('[data-draft-range],[data-draft-number]').forEach(function(input){input.addEventListener('input',function(){var key=input.dataset.draftRange||input.dataset.draftNumber,value=Math.max(0,Math.min(1,parseFloat(input.value)/100));if(!Number.isFinite(value)){return;}state.draft[key]=value;var other=root.querySelector(input.dataset.draftRange?'[data-draft-number="'+key+'"]':'[data-draft-range="'+key+'"]');if(other){other.value=String(Math.round(value*10000)/100);}});});
    var apply=root.querySelector('[data-apply]');if(apply){apply.addEventListener('click',applyThresholds);}
    var reset=root.querySelector('[data-reset]');if(reset){reset.addEventListener('click',function(){state.draft=Object.assign({},DEFAULTS);applyThresholds(true);root.querySelector('[data-reset]').focus({preventScroll:true});});}
    var search=root.querySelector('.search');if(search){search.addEventListener('input',function(){state.search=search.value;state.page=1;refreshQueue();});}
    var reportSearch=root.querySelector('.report-search');if(reportSearch){reportSearch.addEventListener('input',function(){state.reportSearch=reportSearch.value;root.querySelector('.report-models').innerHTML=reportRowsHTML();});}
    var reportsOnly=root.querySelector('[data-reports-only]');if(reportsOnly){reportsOnly.addEventListener('change',function(){state.reportsOnly=reportsOnly.checked;root.querySelector('.report-models').innerHTML=reportRowsHTML();});}
    root.querySelectorAll('[data-filter]').forEach(function(btn){btn.addEventListener('click',function(){state.filter=btn.dataset.filter;render();});});
    if(state.tab==='review'){wireReviewTable(root);}
    else{wireCandidateActions(root);}
    root.querySelectorAll('[data-review-filter]').forEach(function(control){control.addEventListener('change',function(){state[control.dataset.reviewFilter]=control.value;state.page=1;refreshQueue();});});
    var crossOnly=root.querySelector('[data-cross-only]');if(crossOnly){crossOnly.addEventListener('change',function(){setComparisonScope(crossOnly.checked);render();root.querySelector('[data-cross-only]').focus({preventScroll:true});});}
    var back=root.querySelector('[data-back]');if(back){back.addEventListener('click',backToResults);}
    var dialog=root.querySelector('#sms-help'),helpTrigger=null;
    root.onclick=function(event){var button=event.target.closest('[data-help]');if(button&&root.contains(button)){event.preventDefault();helpTrigger=button;openHelp(dialog,button);}};
    dialog.querySelector('[data-help-close]').addEventListener('click',function(){dialog.close();});
    dialog.addEventListener('keydown',function(event){if(event.key==='Escape'){event.preventDefault();dialog.close();}});
    dialog.addEventListener('close',function(){if(helpTrigger&&helpTrigger.isConnected){helpTrigger.focus({preventScroll:true});}});
    wireGroupControls(root);
    var groupSearch=root.querySelector('[data-group-search]');if(groupSearch){groupSearch.addEventListener('input',function(){state.groupSearch=groupSearch.value;refreshGroups();});}
    root.querySelectorAll('[data-compare-select]').forEach(function(sel){sel.addEventListener('change',function(){if(sel.dataset.compareSelect==='a'){state.cmpA=sel.value;}else{state.cmpB=sel.value;}render();});});
    var swap=root.querySelector('[data-swap]');if(swap){swap.addEventListener('click',function(){var tmp=state.cmpA;state.cmpA=state.cmpB;state.cmpB=tmp;render();});}
    var only=root.querySelector('[data-diff-only]');if(only){only.addEventListener('change',function(){state.cmpDiffOnly=only.checked;render();});}
    root.querySelectorAll('[data-section]').forEach(function(button){button.addEventListener('click',function(){var id=button.dataset.section;state.openSections[id]=!state.openSections[id];render();root.querySelector('[data-section="'+id+'"]').focus();});});
    wireMapResults(root);
    var mapSearch=root.querySelector('[data-map-search]');if(mapSearch){mapSearch.addEventListener('input',function(){state.mapSearch=mapSearch.value;state.mapPage=1;refreshMap();});}
    var mapWorkspace=root.querySelector('[data-map-workspace]');if(mapWorkspace){mapWorkspace.addEventListener('change',function(){state.mapWorkspace=mapWorkspace.value;state.mapPage=1;refreshMap();});}
  }

  render();
})();
</script>
"""

    app_json = json.dumps(app_data, allow_nan=False).replace("<", "\\u003c")
    displayHTML(app_template.replace("__APP_DATA__", app_json))


In [ ]:
render_results()